# LeetCode 4081 — Prime Subset Selection II (Hard)

Notebook này giảng bài **Prime Subset Selection II** (Weekly Contest 523 – câu 4, độ khó **Hard**, rating ≈ 2104) theo đúng nhịp "từ căn bản đi lên":

> hiểu đề → tính tay → nền tảng Python & thuật toán → brute force → bước đột phá → lời giải tối ưu → mô phỏng → code nộp LeetCode → bài tập.

Khung code (skeleton) dạng LeetCode:

```python
class Solution:
    def primeSubset(self, n: int, s: int) -> List[int]:
        ...
```

> **Lưu ý trung thực:** trang đề mà notebook dựa vào (bản mirror công khai) **không hiển thị khung code gốc**. Tên phương thức `primeSubset` ở đây là tên tự đặt theo kiểu camelCase. Khi nộp bài, hãy **giữ nguyên tên phương thức trong khung LeetCode cho sẵn** và chỉ dán phần thân hàm vào.

Kỹ thuật trung tâm của bài: **Quy hoạch động (Dynamic Programming – DP) kiểu "cái túi 0/1" (0/1 knapsack) theo tổng**, kết hợp với **dựng lại đáp án nhỏ nhất theo thứ tự từ điển** (lexicographically smallest) bằng một bảng DP trên **hậu tố**. Đây là pattern rất hay gặp trong các bài Hard có câu *"if multiple answers exist, return the lexicographically smallest one"*.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc một đề có **nhiều tiêu chí ưu tiên** (tổng lớn nhất → ít phần tử nhất → nhỏ nhất theo từ điển) và biến chúng thành **một khóa so sánh tuple** duy nhất.
2. Sinh mọi số nguyên tố `≤ n` bằng **sàng Eratosthenes** (Sieve of Eratosthenes).
3. Liệt kê mọi tập con bằng **bitmask** và hiểu vì sao cách này bùng nổ theo `2^m`.
4. Ý tưởng **DP theo tổng** (subset sum / 0/1 knapsack) và lý do bắt buộc duyệt tổng **từ lớn về nhỏ**.
5. Cách **lưu thẳng đáp án tốt nhất vào mỗi trạng thái DP** (Cách 2) và vì sao điều đó đúng.
6. Kỹ thuật **DP trên hậu tố + dựng đáp án tham lam từ trái sang phải** để được mảng nhỏ nhất theo từ điển (Cách 3).
7. Nhận diện pattern và tránh các lỗi kinh điển: duyệt xuôi làm dùng một số hai lần, truy vết ngược ra đáp án "đúng tổng nhưng sai thứ tự từ điển", so sánh chuỗi thay vì so sánh list...

## Các cách giải sẽ học

| Cách | Ý tưởng chính | Độ phức tạp thời gian |
|---|---|---|
| **Cách 1 — Brute Force** | Thử mọi tập con của các số nguyên tố, giữ tập tốt nhất | `O(2^m · m)` |
| **Cách 2 — DP lưu cả đáp án** | `best[t]` = tập tốt nhất có tổng **đúng bằng** `t` | `O(m · s · K)` |
| **Cách 3 — DP hậu tố + dựng đáp án tham lam** | `suf[i][t]` = số phần tử ít nhất; rồi chọn từ trái sang phải | `O(m · s)` |

Ký hiệu dùng xuyên suốt: `m` = số lượng số nguyên tố `≤ n` (tối đa 168), `s` = giới hạn tổng (tối đa 1000), `K` = độ dài đáp án (tối đa 24 — sẽ chứng minh ở mục 1).

# 1. Đề bài (tiếng Việt)

Cho hai số nguyên `n` và `s`.

Xét **tất cả các số nguyên tố nhỏ hơn hoặc bằng `n`**. Hãy chọn một **tập con** (subset) các số nguyên tố này sao cho:

- Mỗi số nguyên tố được chọn **tối đa một lần**.
- **Tổng** các số được chọn **lớn nhất có thể** nhưng vẫn **nhỏ hơn hoặc bằng `s`**.
- Trong các tập con có tổng lớn nhất đó, chọn tập chứa **ít số nguyên tố nhất**.
- Nếu vẫn còn nhiều tập hòa nhau, chọn tập mà khi viết các phần tử theo **thứ tự tăng dần** sẽ tạo thành mảng **nhỏ nhất theo thứ tự từ điển** (lexicographically smallest).

Trả về mảng số nguyên chứa các số nguyên tố đã chọn theo **thứ tự tăng dần**. Nếu **không tồn tại tập con khác rỗng hợp lệ**, trả về **mảng rỗng**.

---

## Ví dụ 1

```text
Input:  n = 7, s = 20
Output: [2, 3, 5, 7]
```

Các số nguyên tố `≤ 7` là `2, 3, 5, 7`. Tổng của tất cả là `2 + 3 + 5 + 7 = 17 ≤ 20`, nên đáp án là `[2, 3, 5, 7]`.

## Ví dụ 2

```text
Input:  n = 15, s = 18
Output: [5, 13]
```

Các số nguyên tố `≤ 15` là `2, 3, 5, 7, 11, 13`. Các tập con có tổng `18` là `[5, 13]`, `[7, 11]`, `[2, 3, 13]`, `[2, 5, 11]`. Vì tổng này bằng đúng `s` nên nó là tổng lớn nhất có thể. Trong số đó, `[5, 13]` và `[7, 11]` ít phần tử nhất (2 phần tử). Giữa hai tập này, `[5, 13]` nhỏ hơn theo từ điển vì phần tử đầu `5 < 7`. Đáp án `[5, 13]`.

## Ví dụ 3

```text
Input:  n = 5, s = 1
Output: []
```

Số nguyên tố nhỏ nhất là `2`, đã lớn hơn `s = 1`. Không chọn được số nào → trả về mảng rỗng.

---

## Ràng buộc (constraints)

- `1 <= n <= 1000`
- `1 <= s <= 1000`

## Ràng buộc nói gì về độ phức tạp cần có?

Đọc constraints **trước khi** nghĩ thuật toán là thói quen bắt buộc với bài Hard.

1. `n ≤ 1000` → chỉ có **168** số nguyên tố `≤ 1000` (sẽ kiểm chứng bằng code ở mục Nền tảng). Gọi số này là `m`.
2. Số tập con của `m` phần tử là `2^m`. Với `m = 168`: `2^168 ≈ 3.7 · 10^50` → **không thể** thử hết, dù máy nhanh đến đâu.
3. Nhưng `s ≤ 1000` **rất nhỏ**! Mọi tổng ta quan tâm đều nằm trong `0..s`, tức chỉ có tối đa `1001` giá trị khác nhau. Đây là tín hiệu kinh điển của **DP theo tổng**: số trạng thái khoảng `m × (s + 1) ≈ 168 × 1001 ≈ 1.7 · 10^5` — cực nhỏ, Python chạy trong vài chục mili giây.
4. Tổng của **25 số nguyên tố nhỏ nhất** đã là `1060 > 1000`, còn tổng của 24 số nhỏ nhất là `963`. Vì vậy **mọi tập con có tổng `≤ s` chứa tối đa 24 số** → đáp án luôn là một mảng ngắn (`K ≤ 24`).

**Kết luận:** ta nhắm tới thuật toán khoảng `O(m · s)`; brute force chỉ dùng để hiểu đề và kiểm tra chéo trên input nhỏ.

In [1]:
def is_prime_simple(x):
    """Kiểm tra x có phải số nguyên tố bằng cách thử chia (chỉ dùng để xem dữ liệu)."""
    if x < 2:
        return False
    for d in range(2, x):
        if x % d == 0:
            return False
    return True


examples = [
    (7, 20, [2, 3, 5, 7]),
    (15, 18, [5, 13]),
    (5, 1, []),
]

for n, s, expected in examples:
    primes = []
    for x in range(2, n + 1):
        if is_prime_simple(x):
            primes.append(x)
    print(f"n = {n:>2}, s = {s:>2} | số nguyên tố ≤ n: {primes} | tổng tất cả = {sum(primes):>2} | đáp án đề: {expected}")

n =  7, s = 20 | số nguyên tố ≤ n: [2, 3, 5, 7] | tổng tất cả = 17 | đáp án đề: [2, 3, 5, 7]
n = 15, s = 18 | số nguyên tố ≤ n: [2, 3, 5, 7, 11, 13] | tổng tất cả = 41 | đáp án đề: [5, 13]
n =  5, s =  1 | số nguyên tố ≤ n: [2, 3, 5] | tổng tất cả = 10 | đáp án đề: []


# 2. Hiểu đề bằng tay trước khi nghĩ thuật toán

Đề có **3 tiêu chí xếp theo thứ tự ưu tiên**. Hãy hình dung như 3 "vòng loại" liên tiếp:

| Vòng | Tiêu chí | Ai được đi tiếp? |
|---|---|---|
| 1 | Tổng **lớn nhất** nhưng `≤ s` | Chỉ các tập có tổng bằng tổng lớn nhất `T` |
| 2 | **Ít phần tử nhất** | Chỉ các tập có số phần tử nhỏ nhất `K` |
| 3 | **Nhỏ nhất theo từ điển** (viết tăng dần) | Đúng 1 tập — đó là đáp án |

Tiêu chí sau **chỉ dùng để phá hòa** của tiêu chí trước, không bao giờ "lật" kết quả của tiêu chí trước.

## Tính tay Ví dụ 2: `n = 15, s = 18`

Số nguyên tố: `2, 3, 5, 7, 11, 13`. Tổng tất cả là `41 > 18`, nên không lấy hết được.

Tổng `18 = s` có tạo được không? Có — liệt kê các tập có tổng 18:

| Tập con (tăng dần) | Tổng | Số phần tử | Qua vòng 1? | Qua vòng 2? | Qua vòng 3? |
|---|---:|---:|:---:|:---:|:---:|
| `[5, 13]` | 18 | 2 | ✔ | ✔ | ✔ (5 < 7) |
| `[7, 11]` | 18 | 2 | ✔ | ✔ | ✘ |
| `[2, 3, 13]` | 18 | 3 | ✔ | ✘ | – |
| `[2, 5, 11]` | 18 | 3 | ✔ | ✘ | – |

Chú ý: `[2, 3, 13]` "nhỏ hơn" `[5, 13]` theo từ điển (vì `2 < 5`) nhưng vẫn **bị loại ở vòng 2** vì có 3 phần tử. Thứ tự vòng loại rất quan trọng.

## Thêm hai ví dụ tự đặt để thấy các "bẫy" của đề

### Ví dụ A: `n = 7, s = 8`

Số nguyên tố: `2, 3, 5, 7`.

- Có tập nào tổng đúng `8` không? `8` không phải số nguyên tố → không có tập 1 phần tử. Tập 2 phần tử: `3 + 5 = 8` ✔.
- Vậy `T = 8`, đáp án `[3, 5]`.

**Bẫy:** nếu "tham lam" lấy số lớn nhất `7` trước thì còn lại `1`, không lấy thêm được số nào → tổng chỉ là `7 < 8`. **Lấy số lớn trước không đảm bảo tổng lớn nhất.**

### Ví dụ B: `n = 5, s = 6`

Số nguyên tố: `2, 3, 5`. Các tổng có thể: `2, 3, 5, 2+3=5, 2+5=7, 3+5=8, 2+3+5=10`.

- Tổng `6` **không tạo được** → tổng lớn nhất `≤ 6` là `T = 5`.
- Có hai tập tổng 5: `[5]` (1 phần tử) và `[2, 3]` (2 phần tử) → chọn `[5]`.

**Bài học:**
1. Tổng lớn nhất `T` có thể **nhỏ hơn `s`** — không được giả sử `T = s`.
2. "Ít phần tử" được xét **trước** "từ điển": `[2, 3]` nhỏ hơn `[5]` theo từ điển nhưng vẫn thua.

In [2]:
print("Ví dụ 2, các tổng bằng 18:", 5 + 13, 7 + 11, 2 + 3 + 13, 2 + 5 + 11)
print("Ví dụ A: 3 + 5 =", 3 + 5, "| tham lam lấy 7 trước chỉ được tổng", 7)
print("Ví dụ B: mọi tổng có thể của {2, 3, 5}:", sorted({2, 3, 5, 2 + 3, 2 + 5, 3 + 5, 2 + 3 + 5}))
print("Ví dụ B: 6 có tạo được không?", 6 in {2, 3, 5, 2 + 3, 2 + 5, 3 + 5, 2 + 3 + 5})

Ví dụ 2, các tổng bằng 18: 18 18 18 18
Ví dụ A: 3 + 5 = 8 | tham lam lấy 7 trước chỉ được tổng 7
Ví dụ B: mọi tổng có thể của {2, 3, 5}: [2, 3, 5, 7, 8, 10]
Ví dụ B: 6 có tạo được không? False


# 3. Nền tảng (kiến thức cần trước khi giải)

Các lời giải phía sau dùng những công cụ sau. Ta học từng cái theo nhịp: **nó là gì → ví dụ nhỏ → code chạy thử → liên quan gì tới bài**.

| Mục | Công cụ | Dùng ở đâu trong bài |
|---|---|---|
| 3.1 | Số nguyên tố, thử chia đến `√x` | Hiểu dữ liệu đầu vào |
| 3.2 | Sàng Eratosthenes, list comprehension | Sinh mọi số nguyên tố `≤ n` trong `O(n log log n)` |
| 3.3 | So sánh list/tuple theo thứ tự từ điển, `min(..., key=...)` | Mã hóa 3 tiêu chí của đề |
| 3.4 | Bitmask: `1 << m`, `(mask >> i) & 1` | Brute force liệt kê mọi tập con |
| 3.5 | DP subset sum / 0/1 knapsack | Trái tim của Cách 2 và Cách 3 |
| 3.6 | Bitset bằng số nguyên Python | Tìm nhanh các tổng tạo được |
| 3.7 | Đệ quy + ghi nhớ (`lru_cache`), `float('inf')` | Góc nhìn top-down của Cách 3 |

## 3.1 Số nguyên tố là gì? Kiểm tra bằng cách thử chia đến `√x`

**Số nguyên tố** (prime number) là số tự nhiên `≥ 2` chỉ chia hết cho `1` và chính nó.

- `2, 3, 5, 7, 11, 13, ...` là số nguyên tố.
- `1` **không** phải số nguyên tố (rất hay bị nhầm!).
- `9 = 3 · 3`, `25 = 5 · 5`, `1000 = 2 · 500` không phải số nguyên tố.

Cách kiểm tra đơn giản: thử chia `x` cho `d = 2, 3, 4, ...`. Ta **chỉ cần thử đến `√x`**, tức là khi `d * d <= x`. Vì sao? Nếu `x = a · b` với `a ≤ b` thì `a · a ≤ a · b = x`, nên ước nhỏ hơn `a` luôn `≤ √x`. Nếu không có ước nào `≤ √x` thì cũng không có ước nào khác.

Ví dụ `x = 29`: thử `d = 2, 3, 4, 5` (vì `6 · 6 = 36 > 29` thì dừng) → không chia hết → `29` là số nguyên tố.

In [3]:
def is_prime_sqrt(x):
    if x < 2:
        return False
    d = 2
    while d * d <= x:          # chỉ thử ước đến căn bậc hai
        if x % d == 0:
            return False
        d += 1
    return True


for x in [1, 2, 9, 13, 25, 29, 997, 1000]:
    print(f"{x:>4} -> {is_prime_sqrt(x)}")

   1 -> False
   2 -> True
   9 -> False
  13 -> True
  25 -> False
  29 -> True
 997 -> True
1000 -> False


## 3.2 Sàng Eratosthenes — sinh mọi số nguyên tố `≤ n` một lần

Kiểm tra từng số một thì lặp lại nhiều việc. **Sàng Eratosthenes** làm ngược lại: thay vì hỏi "x có ước không?", ta **gạch bỏ các bội** của mỗi số nguyên tố.

Tính tay với `n = 30`:

1. Viết `2, 3, ..., 30`, ban đầu coi tất cả là "có thể nguyên tố".
2. `2` chưa bị gạch → nguyên tố. Gạch `4, 6, 8, ..., 30`.
3. `3` chưa bị gạch → nguyên tố. Gạch `9, 12, 15, ..., 30` (bắt đầu từ `3 · 3 = 9`, vì `6` đã bị `2` gạch rồi).
4. `4` đã bị gạch → bỏ qua. `5` chưa bị gạch → gạch `25, 30`.
5. `6 · 6 = 36 > 30` → dừng. Những số còn lại là số nguyên tố: `2, 3, 5, 7, 11, 13, 17, 19, 23, 29`.

Hai chi tiết quan trọng:
- Gạch từ `i * i` (không phải từ `2 * i`) vì các bội nhỏ hơn đã bị số nguyên tố nhỏ hơn gạch.
- Mảng có kích thước `n + 1` để **chứa cả chỉ số `n`** (nếu `n` là số nguyên tố, ví dụ `n = 7`, thì `7` phải có mặt!).

**List comprehension** `[x for x in range(n + 1) if is_prime[x]]` là cách viết gọn của: "duyệt `x`, nếu `is_prime[x]` đúng thì thêm `x` vào list".

Độ phức tạp của sàng: `O(n log log n)` — gần như tuyến tính.

In [4]:
def sieve_primes(n):
    """Trả về list các số nguyên tố <= n (sàng Eratosthenes), theo thứ tự tăng dần."""
    if n < 2:
        return []
    is_prime = [True] * (n + 1)          # is_prime[x]: x còn "sống sót" sau khi sàng không
    is_prime[0] = is_prime[1] = False    # 0 và 1 không phải số nguyên tố
    i = 2
    while i * i <= n:
        if is_prime[i]:
            for j in range(i * i, n + 1, i):   # gạch các bội của i, bắt đầu từ i*i
                is_prime[j] = False
        i += 1
    return [x for x in range(n + 1) if is_prime[x]]


print("Số nguyên tố ≤ 30:", sieve_primes(30))
print("Có bao nhiêu số nguyên tố ≤ 1000:", len(sieve_primes(1000)))
print("Sàng khớp với cách thử chia cho mọi x ≤ 1000:",
      sieve_primes(1000) == [x for x in range(1001) if is_prime_sqrt(x)])

P = sieve_primes(1000)
print("Tổng 24 số nguyên tố nhỏ nhất:", sum(P[:24]), "| tổng 25 số nhỏ nhất:", sum(P[:25]))

Số nguyên tố ≤ 30: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
Có bao nhiêu số nguyên tố ≤ 1000: 168
Sàng khớp với cách thử chia cho mọi x ≤ 1000: True
Tổng 24 số nguyên tố nhỏ nhất: 963 | tổng 25 số nhỏ nhất: 1060


**Liên hệ với bài:** bước đầu tiên của mọi lời giải là `primes = sieve_primes(n)`. Kết quả đã **tăng dần** — điều này sẽ cực kỳ quan trọng khi dựng đáp án theo thứ tự từ điển.

Dòng cuối cũng xác nhận nhận xét ở mục 1: `963 ≤ 1000 < 1060` nên một tập có tổng `≤ 1000` chứa **tối đa 24** số nguyên tố.

## 3.3 So sánh list/tuple theo thứ tự từ điển và khóa nhiều tiêu chí

Python so sánh hai list (hoặc tuple) theo **thứ tự từ điển** (lexicographic order), giống tra từ điển:

1. So phần tử đầu tiên. Khác nhau → bên nào nhỏ hơn thì cả list nhỏ hơn. **Dừng.**
2. Bằng nhau → so tiếp phần tử thứ hai, thứ ba...
3. Nếu một list là **tiền tố** (prefix) của list kia (`[3, 5]` và `[3, 5, 7]`) thì list **ngắn hơn** nhỏ hơn.

Ví dụ: `[5, 13] < [7, 11]` vì `5 < 7`; `[5, 13] < [5, 17]` vì `5 == 5` rồi `13 < 17`.

**Cẩn thận:** `[2, 3, 13] < [5, 13]` là `True` (vì `2 < 5`) dù list bên trái **dài hơn**. Thứ tự từ điển thuần túy **không quan tâm độ dài trước**. Đó là lý do đề cần tiêu chí "ít phần tử" riêng.

In [5]:
print([5, 13] < [7, 11])          # 5 < 7 quyết định ngay
print([5, 13] < [5, 17])          # 5 == 5, so tiếp 13 < 17
print([2, 3, 13] < [5, 13])       # 2 < 5 -> list DÀI hơn vẫn "nhỏ hơn"!
print([3, 5] < [3, 5, 7])         # tiền tố thì nhỏ hơn
print((2, [5, 13]) < (2, [7, 11]))      # tuple: 2 == 2 rồi so list
print((2, [7, 11]) < (3, [2, 3, 13]))   # tuple: 2 < 3 quyết định, không cần nhìn list

True
True
True
True
True
True


### Mã hóa cả 3 tiêu chí thành MỘT khóa tuple

Tuple cũng so sánh từ điển: so thành phần thứ nhất, hòa thì so thành phần thứ hai... Đây chính xác là cơ chế "vòng loại" ở mục 2! Ta xây **khóa xếp hạng** (ranking key) sao cho **khóa nhỏ hơn = tập tốt hơn**:

```python
key = (-tổng, số_phần_tử, mảng_tăng_dần)
```

- `-tổng`: ta muốn tổng **lớn nhất**, nhưng `min` chọn cái **nhỏ nhất** → đổi dấu. Tổng 18 cho `-18`, tổng 17 cho `-17`, và `-18 < -17`.
- `số_phần_tử`: muốn **ít nhất** → để nguyên.
- `mảng_tăng_dần`: muốn **nhỏ nhất theo từ điển** → để nguyên.

Hàm `min(danh_sách, key=f)` trả về phần tử có `f(phần_tử)` nhỏ nhất. `f` có thể là một hàm `def` bình thường hoặc `lambda` (hàm vô danh một dòng, ví dụ `lambda c: (len(c), c)`).

In [6]:
s = 18
candidates = [[2, 3, 13], [7, 11], [5, 13], [2, 5, 11], [2, 3, 5, 7], [7, 13]]
valid = [c for c in candidates if sum(c) <= s]      # [7, 13] có tổng 20 > 18 bị loại


def rank_key(c):
    return (-sum(c), len(c), c)


for c in valid:
    print(f"{str(c):<14} -> key = {rank_key(c)}")
print("Tập tốt nhất theo khóa:", min(valid, key=rank_key))

[2, 3, 13]     -> key = (-18, 3, [2, 3, 13])
[7, 11]        -> key = (-18, 2, [7, 11])
[5, 13]        -> key = (-18, 2, [5, 13])
[2, 5, 11]     -> key = (-18, 3, [2, 5, 11])
[2, 3, 5, 7]   -> key = (-17, 4, [2, 3, 5, 7])
Tập tốt nhất theo khóa: [5, 13]


**Liên hệ với bài:** khóa `(-tổng, số_phần_tử, mảng)` là "định nghĩa chính thức" của chữ "tốt hơn" trong đề. Brute force dùng thẳng khóa này; Cách 2 dùng phần `(số_phần_tử, mảng)` cho từng tổng cố định.

## 3.4 Bitmask — liệt kê mọi tập con

Với `m` phần tử, mỗi tập con tương ứng với một dãy `m` bit: bit thứ `i` bằng `1` nghĩa là **chọn** phần tử thứ `i`. Mọi dãy bit như vậy chính là các số nguyên từ `0` đến `2^m − 1`.

- `1 << m` nghĩa là "dịch số 1 sang trái `m` bit" = `2^m`. Ví dụ `1 << 3 = 8`.
- `(mask >> i) & 1` lấy bit thứ `i` của `mask` (đếm từ bên phải, bắt đầu từ 0).

Ví dụ với `primes = [2, 3, 5]`: `mask = 6 = 110` (nhị phân) → bit 0 = 0, bit 1 = 1, bit 2 = 1 → chọn `primes[1] = 3` và `primes[2] = 5` → tập `[3, 5]`.

In [7]:
primes = [2, 3, 5]
m = len(primes)
for mask in range(1 << m):                       # 1 << 3 = 8 -> mask = 0..7
    chosen = [primes[i] for i in range(m) if (mask >> i) & 1]
    print(f"mask = {mask} = {mask:03b} (nhị phân, đọc bit từ phải) -> {chosen}, tổng = {sum(chosen)}")

mask = 0 = 000 (nhị phân, đọc bit từ phải) -> [], tổng = 0
mask = 1 = 001 (nhị phân, đọc bit từ phải) -> [2], tổng = 2
mask = 2 = 010 (nhị phân, đọc bit từ phải) -> [3], tổng = 3
mask = 3 = 011 (nhị phân, đọc bit từ phải) -> [2, 3], tổng = 5
mask = 4 = 100 (nhị phân, đọc bit từ phải) -> [5], tổng = 5
mask = 5 = 101 (nhị phân, đọc bit từ phải) -> [2, 5], tổng = 7
mask = 6 = 110 (nhị phân, đọc bit từ phải) -> [3, 5], tổng = 8
mask = 7 = 111 (nhị phân, đọc bit từ phải) -> [2, 3, 5], tổng = 10


**Liên hệ với bài:** brute force duyệt `mask` từ `1` đến `2^m − 1` (bỏ `mask = 0` là tập rỗng). Vì ta duyệt `i` tăng dần và `primes` tăng dần, `chosen` **tự động tăng dần** — không cần sort.

## 3.5 DP subset sum / "cái túi 0/1" — tổng nào tạo được?

Câu hỏi: với các số `nums`, những tổng `t` nào trong `0..s` tạo được bằng cách chọn **mỗi số tối đa một lần**?

Ý tưởng **Quy hoạch động (DP)**: thay vì liệt kê tập con, ta lưu một mảng `can[t]` = "tổng `t` đã tạo được chưa".

- Khởi tạo: `can[0] = True` (chọn tập rỗng), các ô khác `False`.
- Thêm từng số `p`: nếu trước đó tạo được `t − p` thì giờ tạo được `t` (thêm `p` vào).

**Vì sao phải duyệt `t` từ lớn về nhỏ?** Để mỗi `p` chỉ được dùng **một lần**. Ví dụ `nums = [2]`, `s = 4`:

- Duyệt **xuôi** `t = 2, 3, 4`: `can[2] = can[0] = True`, rồi `can[4] = can[2] = True` — nhưng `can[2]` vừa được bật **bởi chính số 2** → ta đã dùng `2` hai lần (`2 + 2 = 4`). **Sai.**
- Duyệt **ngược** `t = 4, 3, 2`: `can[4] = can[2]` (lúc này `can[2]` còn là giá trị **cũ** `False`) → đúng; sau đó mới `can[2] = True`.

Quy tắc nhớ: **0/1 knapsack (mỗi món ≤ 1 lần) → duyệt tổng ngược; unbounded knapsack (dùng bao nhiêu lần cũng được) → duyệt xuôi.**

In [8]:
def reachable_sums(nums, s, backward=True):
    can = [False] * (s + 1)
    can[0] = True                                  # tổng 0: chọn tập rỗng
    for p in nums:
        order = range(s, p - 1, -1) if backward else range(p, s + 1)
        for t in order:
            if can[t - p]:
                can[t] = True
    return can


print("nums=[2], s=4, duyệt ngược:", reachable_sums([2], 4, backward=True))
print("nums=[2], s=4, duyệt xuôi :", reachable_sums([2], 4, backward=False), "<- can[4] sai (dùng 2 hai lần)")

can = reachable_sums([2, 3, 5, 7, 11, 13], 41)
print("Số nguyên tố ≤ 15, tổng KHÔNG tạo được trong 1..41:", [t for t in range(1, 42) if not can[t]])

nums=[2], s=4, duyệt ngược: [True, False, True, False, False]
nums=[2], s=4, duyệt xuôi : [True, False, True, False, True] <- can[4] sai (dùng 2 hai lần)
Số nguyên tố ≤ 15, tổng KHÔNG tạo được trong 1..41: [1, 4, 6, 35, 37, 40]


**Liên hệ với bài:** chỉ riêng mảng `can` đã giải được **tiêu chí 1**: tổng lớn nhất `T` là `t` lớn nhất `≤ s` với `can[t] = True`. Ví dụ với số nguyên tố `≤ 15` và `s = 6`, `can[6] = False` nên `T = 5` — đúng như Ví dụ B.

## 3.6 Bitset bằng số nguyên Python

Số nguyên Python có thể dài tùy ý, nên ta dùng **từng bit** của một số `bits` thay cho mảng `can`: bit thứ `t` bằng `1` ⇔ tổng `t` tạo được.

- `bits << p`: dịch mọi bit sang trái `p` vị trí ⇔ "mọi tổng cũ cộng thêm `p`".
- `bits |= bits << p`: hợp (OR) tập tổng cũ với tập tổng mới. Vì vế phải được tính **trước** khi gán, mỗi `p` tự động chỉ dùng một lần.

Đây là cùng thuật toán với 3.5, nhưng xử lý hàng chục bit mỗi phép toán nên rất nhanh.

In [9]:
bits = 1                                  # chỉ bit 0 bật: tổng 0 tạo được
for p in [2, 3, 5, 7, 11, 13]:
    bits |= bits << p                     # thêm p vào mọi tổng cũ

sums = [t for t in range(42) if (bits >> t) & 1]
print("Các tổng tạo được:", sums)

s = 18
T = max(t for t in range(s + 1) if (bits >> t) & 1)
print("Với s = 18, tổng lớn nhất T =", T)

Các tổng tạo được: [0, 2, 3, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 36, 38, 39, 41]
Với s = 18, tổng lớn nhất T = 18


**Liên hệ với bài:** bitset trả lời rất nhanh "tổng nào tạo được" (tiêu chí 1), nhưng **không** cho biết số phần tử ít nhất hay tập nào nhỏ nhất theo từ điển. Ta sẽ cần DP "giàu thông tin" hơn.

## 3.7 Đệ quy + ghi nhớ (`lru_cache`) và `float('inf')`

Nhiều bài DP dễ nghĩ nhất dưới dạng **đệ quy**: "đứng ở phần tử thứ `i`, còn cần tổng `t`, có bao nhiêu cách?" — hoặc **chọn** `primes[i]`, hoặc **bỏ qua** nó.

Đệ quy thuần sẽ tính lại cùng một cặp `(i, t)` rất nhiều lần. `@lru_cache(maxsize=None)` (trong `functools`) **ghi nhớ** (memoization) kết quả theo tham số: lần sau gặp lại `(i, t)` thì trả ngay. Khi đó số phép tính chỉ bằng **số trạng thái khác nhau**.

`float('inf')` là "vô cực": lớn hơn mọi số, `INF + 1 == INF`, `min(INF, 3) == 3`. Ta dùng nó cho ý nghĩa **"không thể"** khi tìm giá trị nhỏ nhất.

In [10]:
from functools import lru_cache

primes15 = [2, 3, 5, 7, 11, 13]


@lru_cache(maxsize=None)
def count_ways(i, t):
    """Số tập con của primes15[i:] có tổng ĐÚNG bằng t."""
    if t == 0:
        return 1                      # đã đủ tổng: 1 cách (không chọn thêm gì)
    if t < 0 or i == len(primes15):
        return 0                      # vượt tổng hoặc hết số
    skip = count_ways(i + 1, t)                   # bỏ qua primes15[i]
    take = count_ways(i + 1, t - primes15[i])     # chọn primes15[i]
    return skip + take


print("Số tập con của {2,3,5,7,11,13} có tổng 18:", count_ways(0, 18))

INF = float('inf')
print("INF + 1 == INF:", INF + 1 == INF, "| min(INF, 3) =", min(INF, 3), "| 1000 < INF:", 1000 < INF)

Số tập con của {2,3,5,7,11,13} có tổng 18: 4
INF + 1 == INF: True | min(INF, 3) = 3 | 1000 < INF: True


**Liên hệ với bài:** kết quả `4` khớp đúng 4 tập con tổng 18 trong Ví dụ 2. Ở Cách 3, ta sẽ dùng **cùng khung "chọn / bỏ qua `primes[i]`"** nhưng thay "đếm số cách" bằng "**số phần tử ít nhất**" (dùng `min` và `INF`).

---

# 4. Cách 1 — Brute Force: thử mọi tập con

## Ý tưởng

Đây là cách "đọc đề rồi làm đúng như đề nói":

1. Sinh `primes = sieve_primes(n)`.
2. Duyệt mọi tập con khác rỗng bằng bitmask.
3. Bỏ tập nào có tổng `> s`.
4. Với tập hợp lệ, tính khóa `(-tổng, số_phần_tử, mảng)`; giữ tập có khóa nhỏ nhất.
5. Nếu không có tập hợp lệ nào → trả `[]`.

Cách này **hiển nhiên đúng** theo đề, nên nó là "trọng tài" để kiểm tra các cách nhanh hơn.

In [11]:
from typing import List


def prime_subset_bruteforce(n: int, s: int) -> List[int]:
    primes = sieve_primes(n)
    m = len(primes)
    best_key = None
    best_subset = []
    for mask in range(1, 1 << m):                     # bỏ mask = 0 (tập rỗng)
        chosen = [primes[i] for i in range(m) if (mask >> i) & 1]
        total = sum(chosen)
        if total > s:
            continue                                  # vi phạm tổng <= s
        key = (-total, len(chosen), chosen)           # khóa nhỏ hơn = tập tốt hơn
        if best_key is None or key < best_key:
            best_key = key
            best_subset = chosen
    return best_subset


for n, s, expected in examples:
    got = prime_subset_bruteforce(n, s)
    print(f"n = {n:>2}, s = {s:>2} -> {got}  (mong đợi {expected})  {'OK' if got == expected else 'SAI'}")

n =  7, s = 20 -> [2, 3, 5, 7]  (mong đợi [2, 3, 5, 7])  OK
n = 15, s = 18 -> [5, 13]  (mong đợi [5, 13])  OK
n =  5, s =  1 -> []  (mong đợi [])  OK


## Cách 1 tốn bao nhiêu công?

- Số tập con: `2^m`. Mỗi tập tốn `O(m)` để dựng `chosen` và tính tổng.
- Tổng cộng: **`O(2^m · m)`** thời gian, `O(m)` bộ nhớ.

Bảng dưới cho thấy `2^m` tăng khủng khiếp thế nào khi `n` tăng. Với `n = 1000` (`m = 168`), số tập con có **51 chữ số**. Brute force chỉ dùng được khi `n` khoảng `≤ 50–60`.

In [12]:
import time

for n in [7, 15, 30, 50, 100, 200, 1000]:
    m = len(sieve_primes(n))
    print(f"n = {n:>4} -> m = {m:>3} số nguyên tố -> 2^m = {2 ** m:.3e} tập con")

for n in [30, 40, 50]:
    start = time.perf_counter()
    prime_subset_bruteforce(n, 100)
    ms = (time.perf_counter() - start) * 1000
    print(f"Brute force với n = {n} (m = {len(sieve_primes(n))}) mất khoảng {ms:.0f} ms")

n =    7 -> m =   4 số nguyên tố -> 2^m = 1.600e+01 tập con
n =   15 -> m =   6 số nguyên tố -> 2^m = 6.400e+01 tập con
n =   30 -> m =  10 số nguyên tố -> 2^m = 1.024e+03 tập con
n =   50 -> m =  15 số nguyên tố -> 2^m = 3.277e+04 tập con
n =  100 -> m =  25 số nguyên tố -> 2^m = 3.355e+07 tập con
n =  200 -> m =  46 số nguyên tố -> 2^m = 7.037e+13 tập con
n = 1000 -> m = 168 số nguyên tố -> 2^m = 3.741e+50 tập con
Brute force với n = 30 (m = 10) mất khoảng 1 ms
Brute force với n = 40 (m = 12) mất khoảng 2 ms
Brute force với n = 50 (m = 15) mất khoảng 21 ms


# 5. Bước đột phá

## 5.1 Rất nhiều tập con, nhưng rất ít **tổng** khác nhau

Brute force chậm vì nó phân biệt từng tập con. Nhưng đề chỉ hỏi về **tổng ≤ s ≤ 1000**. Hai tập khác nhau có cùng tổng thì "cạnh tranh trực tiếp" với nhau — ta chỉ cần giữ **tập tốt nhất** cho mỗi tổng!

Thí nghiệm: đếm số tập con và số **tổng khác nhau** (kể cả tổng 0).

In [13]:
for n in [15, 30, 50]:
    primes = sieve_primes(n)
    m = len(primes)
    distinct = set()
    for mask in range(1 << m):
        distinct.add(sum(primes[i] for i in range(m) if (mask >> i) & 1))
    print(f"n = {n:>2}: {2 ** m:>6} tập con nhưng chỉ có {len(distinct):>3} tổng khác nhau (tổng lớn nhất = {sum(primes)})")

n = 15:     64 tập con nhưng chỉ có  36 tổng khác nhau (tổng lớn nhất = 41)
n = 30:   1024 tập con nhưng chỉ có 124 tổng khác nhau (tổng lớn nhất = 129)
n = 50:  32768 tập con nhưng chỉ có 323 tổng khác nhau (tổng lớn nhất = 328)


**Quan sát:** với `n = 50`, có `32768` tập con nhưng chỉ `323` tổng khác nhau. Khi `n = 1000`, số tập con là `~10^50` nhưng số tổng ta quan tâm (`0..s`) **không bao giờ vượt quá 1001**.

→ Hãy dùng **tổng `t` làm trạng thái** (state) của DP. Đây là chìa khóa biến bài toán mũ thành bài toán đa thức giả (pseudo-polynomial) `O(m · s)`.

## 5.2 Tách đề thành 3 câu hỏi nhỏ

| Câu hỏi | Ký hiệu | Công cụ đơn giản nhất |
|---|---|---|
| Tổng lớn nhất `≤ s` tạo được là bao nhiêu? | `T` | mảng `can` / bitset (mục 3.5, 3.6) |
| Với tổng `T`, cần ít nhất bao nhiêu số? | `K` | DP "số phần tử ít nhất" |
| Trong các tập tổng `T`, `K` phần tử, tập nào nhỏ nhất theo từ điển? | đáp án | **cần một kỹ thuật dựng đáp án** |

Hai câu đầu là DP knapsack quen thuộc. **Câu thứ ba mới là phần "Hard"** của bài.

## 5.3 Thử tham lam (greedy) — và thấy nó thất bại

Một ý tưởng tự nhiên: "lấy số lớn nhất còn vừa" (giống trả tiền xu), hoặc "lấy số nhỏ nhất trước cho mảng nhỏ theo từ điển". Hãy kiểm tra bằng thí nghiệm so với brute force.

In [14]:
def greedy_largest_first(n, s):
    ans, rem = [], s
    for p in reversed(sieve_primes(n)):       # số lớn trước
        if p <= rem:
            ans.append(p)
            rem -= p
    return sorted(ans)


def greedy_smallest_first(n, s):
    ans, rem = [], s
    for p in sieve_primes(n):                 # số nhỏ trước
        if p <= rem:
            ans.append(p)
            rem -= p
    return ans


for n, s in [(7, 8), (11, 12), (3, 3), (15, 18)]:
    print(f"n = {n:>2}, s = {s:>2} | lớn trước: {str(greedy_largest_first(n, s)):<14}"
          f"| nhỏ trước: {str(greedy_smallest_first(n, s)):<14}| đúng (brute): {prime_subset_bruteforce(n, s)}")

n =  7, s =  8 | lớn trước: [7]           | nhỏ trước: [2, 3]        | đúng (brute): [3, 5]
n = 11, s = 12 | lớn trước: [11]          | nhỏ trước: [2, 3, 5]     | đúng (brute): [5, 7]
n =  3, s =  3 | lớn trước: [3]           | nhỏ trước: [2]           | đúng (brute): [3]
n = 15, s = 18 | lớn trước: [5, 13]       | nhỏ trước: [2, 3, 5, 7]  | đúng (brute): [5, 13]


Cả hai kiểu tham lam đều sai ở nhiều input (ví dụ `n = 7, s = 8`: lớn-trước ra `[7]` tổng 7, nhỏ-trước ra `[2, 3]` tổng 5, trong khi đáp án đúng `[3, 5]` tổng 8). Chúng **đôi khi đúng nhờ may mắn** (`n = 15, s = 18` với lớn-trước).

Bài học: không có quy tắc chọn cục bộ nào đúng sẵn. Nhưng ý tưởng "**chọn số nhỏ nhất trước**" không vứt đi — ta sẽ sửa nó ở Cách 3 bằng cách **hỏi trước một "nhà tiên tri"**: *"Nếu chọn số này, phần còn lại có còn hoàn thành tối ưu được không?"*

## 5.4 Ý tưởng then chốt cho Cách 2: lưu **tập tốt nhất** cho mỗi tổng

Gọi các số nguyên tố tăng dần là `p_1 < p_2 < ... < p_m`. Ta thêm lần lượt từng số và duy trì:

> `best[t]` = tập tốt nhất (ít phần tử nhất, hòa thì nhỏ nhất theo từ điển) trong các tập con **của những số đã thêm** có tổng **đúng bằng** `t`. Nếu không tạo được, `best[t] = None`.

Khi thêm số mới `p` (lớn hơn mọi số đã thêm), một tập có tổng `t` hoặc:
- **không chứa `p`** → ứng viên tốt nhất là `best[t]` cũ; hoặc
- **chứa `p`** → có dạng `X + [p]` với `X` có tổng `t − p`. Vì `p` lớn nhất, `p` luôn đứng **cuối** mảng tăng dần. So hai tập `X + [p]` và `Y + [p]`: số phần tử hơn kém y như `X` và `Y`; nếu bằng nhau thì so từ điển cũng y như so `X` với `Y` (phần cuối giống nhau). → `X` tốt nhất chính là `best[t − p]` cũ!

Ví dụ số: với Ví dụ 2, sau khi thêm `2, 3, 5, 7, 11` ta có `best[18] = (7, 11)`. Thêm `13`: ứng viên mới là `best[5] + (13,) = (5,) + (13,) = (5, 13)`. So `(2, (5, 13))` với `(2, (7, 11))` → `(5, 13)` thắng. Ta **không cần nhớ** `[2, 3, 13]` hay `[2, 5, 11]` — chúng đã thua ngay từ trạng thái con.

# 6. Cách 2 — DP lưu cả đáp án cho từng tổng

## Từng bước xây dựng

1. **Trạng thái:** `best[t]` là một tuple tăng dần (hoặc `None`), với `t = 0..s`.
2. **Khởi tạo:** `best[0] = ()` (tập rỗng có tổng 0), các ô khác `None`.
3. **Chuyển trạng thái:** với mỗi `p` theo thứ tự **tăng dần**, với `t` từ `s` **về** `p`:
   - `cand = best[t − p] + (p,)` (nếu `best[t − p]` khác `None`);
   - nếu `best[t]` là `None` hoặc `(len(cand), cand) < (len(best[t]), best[t])` thì `best[t] = cand`.
4. **Duyệt `t` ngược** để `best[t − p]` luôn là giá trị **chưa chứa `p`** (mỗi số dùng ≤ 1 lần — mục 3.5).
5. **Kết quả:** tìm `t` lớn nhất trong `s, s−1, ..., 1` có `best[t]` khác `None` → trả `list(best[t])`. Không có → `[]`.

Vì sao dùng **tuple** thay vì list? Tuple không đổi được (immutable), nên `prev + (p,)` tạo tuple mới mà **không làm hỏng** `best[t − p]` đang được dùng chung. So sánh tuple cũng theo từ điển như list.

Vì sao phải so `(len(cand), cand)` chứ không chỉ `cand`? Vì như mục 3.3, `(2, 3, 13) < (5, 13)` nhưng `(2, 3, 13)` có nhiều phần tử hơn — phải so **số phần tử trước**.

In [15]:
def prime_subset_dp_lists(n: int, s: int) -> List[int]:
    primes = sieve_primes(n)
    best = [None] * (s + 1)       # best[t]: tuple tốt nhất có tổng ĐÚNG bằng t, None = không tạo được
    best[0] = ()                  # tổng 0: tập rỗng
    for p in primes:                          # thêm các số nguyên tố theo thứ tự TĂNG DẦN
        for t in range(s, p - 1, -1):         # duyệt tổng TỪ LỚN VỀ NHỎ (mỗi p dùng tối đa 1 lần)
            prev = best[t - p]
            if prev is None:
                continue
            cand = prev + (p,)                # p lớn nhất -> gắn vào cuối vẫn tăng dần
            cur = best[t]
            if cur is None or (len(cand), cand) < (len(cur), cur):
                best[t] = cand
    for t in range(s, 0, -1):                 # tổng lớn nhất (> 0) tạo được
        if best[t] is not None:
            return list(best[t])
    return []                                 # không có tập con khác rỗng hợp lệ


for n, s, expected in examples:
    got = prime_subset_dp_lists(n, s)
    print(f"n = {n:>2}, s = {s:>2} -> {got}  (mong đợi {expected})  {'OK' if got == expected else 'SAI'}")
print("n = 7, s = 8 ->", prime_subset_dp_lists(7, 8), "| n = 5, s = 6 ->", prime_subset_dp_lists(5, 6))

n =  7, s = 20 -> [2, 3, 5, 7]  (mong đợi [2, 3, 5, 7])  OK
n = 15, s = 18 -> [5, 13]  (mong đợi [5, 13])  OK
n =  5, s =  1 -> []  (mong đợi [])  OK
n = 7, s = 8 -> [3, 5] | n = 5, s = 6 -> [5]


## Mô phỏng Cách 2: theo dõi ô `best[18]` của Ví dụ 2

Ta in giá trị `best[18]` sau khi thêm mỗi số nguyên tố, kèm theo ứng viên `best[18 − p] + (p,)` (lấy giá trị **cũ** của `best[18 − p]`, trước khi thêm `p`).

| Thêm `p` | `best[18 − p]` (cũ) | Ứng viên | `best[18]` sau bước | Ghi chú |
|---:|---|---|---|---|
| 2 | `best[16] = None` | – | `None` | chỉ có {2} |
| 3 | `best[15] = None` | – | `None` | |
| 5 | `best[13] = None` | – | `None` | |
| 7 | `best[11] = None` | – | `None` | {2,3,5} chưa tạo được 11 |
| 11 | `best[7] = (7,)` | `(7, 11)` | `(7, 11)` | lần đầu tạo được 18 |
| 13 | `best[5] = (5,)` | `(5, 13)` | `(5, 13)` | cùng 2 phần tử, `5 < 7` → cập nhật |

Lưu ý ở bước `p = 11`: tập `[2, 5, 11]` cũng có tổng 18, nhưng nó **không bao giờ xuất hiện** vì `best[7]` đã chọn `(7,)` (1 phần tử) thay vì `(2, 5)` (2 phần tử).

In [16]:
def trace_best_value(n, s, target):
    """In best[target] sau khi thêm từng số nguyên tố (Cách 2)."""
    primes = sieve_primes(n)
    best = [None] * (s + 1)
    best[0] = ()
    for p in primes:
        old_target = best[target]
        prev = best[target - p] if target >= p else None      # giá trị CŨ, trước khi thêm p
        cand = None if prev is None else prev + (p,)
        for t in range(s, p - 1, -1):
            pv = best[t - p]
            if pv is None:
                continue
            c = pv + (p,)
            cur = best[t]
            if cur is None or (len(c), c) < (len(cur), cur):
                best[t] = c
        note = "cập nhật" if best[target] != old_target else "giữ nguyên"
        print(f"thêm p = {p:>2} | best[{target - p:>2}] cũ = {str(prev):<8} | ứng viên = {str(cand):<9}"
              f"| best[{target}] = {str(best[target]):<9} ({note})")
    return best[target]


trace_best_value(15, 18, 18)

thêm p =  2 | best[16] cũ = None     | ứng viên = None     | best[18] = None      (giữ nguyên)
thêm p =  3 | best[15] cũ = None     | ứng viên = None     | best[18] = None      (giữ nguyên)
thêm p =  5 | best[13] cũ = None     | ứng viên = None     | best[18] = None      (giữ nguyên)
thêm p =  7 | best[11] cũ = None     | ứng viên = None     | best[18] = None      (giữ nguyên)
thêm p = 11 | best[ 7] cũ = (7,)     | ứng viên = (7, 11)  | best[18] = (7, 11)   (cập nhật)
thêm p = 13 | best[ 5] cũ = (5,)     | ứng viên = (5, 13)  | best[18] = (5, 13)   (cập nhật)


(5, 13)

## Độ phức tạp Cách 2

- Có `m` số nguyên tố, mỗi số duyệt tối đa `s` tổng → `O(m · s)` lần cập nhật.
- Mỗi lần cập nhật tạo/so sánh tuple dài tối đa `K ≤ 24` → **Time `O(m · s · K)`**.
- Bộ nhớ: `s + 1` tuple, mỗi tuple dài `≤ K` → **Space `O(s · K)`**.

Với `m = 168, s = 1000, K ≤ 24` thì chạy rất nhanh. Cách 2 **ngắn, đúng, đủ để AC**.

**Vậy tại sao còn Cách 3?**
1. Cách 2 dựa vào việc "cả đáp án" nhỏ và dễ so sánh. Nếu đáp án dài (ví dụ `K` hàng nghìn) hoặc ngôn ngữ không có so sánh tuple sẵn (C++/Java phải tự viết), chi phí `K` trở nên đáng kể.
2. Cách 3 dạy một **kỹ thuật tổng quát**: DP chỉ lưu **số** (rẻ), sau đó **dựng lại đáp án nhỏ nhất theo từ điển**. Kỹ thuật này dùng được cho rất nhiều bài khác (xem mục 13).

# 7. Cách 3 — DP đếm trên hậu tố + dựng đáp án tham lam

## 7.1 Bước A + B: chỉ lưu **số phần tử ít nhất**

Bỏ tuple đi, chỉ giữ một con số:

> `dp[t]` = số phần tử ít nhất để tạo tổng **đúng** `t` (hoặc `INF` nếu không tạo được).

Đây là 0/1 knapsack "đếm ít nhất": `dp[t] = min(dp[t], dp[t − p] + 1)`, duyệt `t` ngược. Từ `dp` ta có ngay:
- `T` = `t` lớn nhất trong `1..s` có `dp[t] < INF` (tiêu chí 1);
- `K = dp[T]` (tiêu chí 2).

In [17]:
INF = float('inf')


def min_count_1d(primes, s):
    dp = [INF] * (s + 1)
    dp[0] = 0
    for p in primes:
        for t in range(s, p - 1, -1):          # 0/1 knapsack -> duyệt ngược
            dp[t] = min(dp[t], dp[t - p] + 1)
    return dp


def fmt(x):
    return '∞' if x == INF else str(x)


primes = sieve_primes(15)
dp = min_count_1d(primes, 18)
print("t    :", ' '.join(f"{t:>2}" for t in range(19)))
print("dp[t]:", ' '.join(f"{fmt(x):>2}" for x in dp))
T = max(t for t in range(1, 19) if dp[t] < INF)
print("T =", T, "| K = dp[T] =", dp[T])

t    :  0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18
dp[t]:  0  ∞  1  1  ∞  1  ∞  1  2  2  2  1  2  1  2  2  2  4  2
T = 18 | K = dp[T] = 2


## 7.2 Vấn đề: biết `T` và `K`, nhưng **tập nào**?

Cách truy vết (backtracking the DP) quen thuộc trong sách là: lưu bảng `pre[i][t]` dùng `i` số **đầu tiên** (tiền tố — prefix), rồi đi **từ số cuối về số đầu**: nếu bỏ được `primes[i−1]` mà vẫn tối ưu thì bỏ, không thì chọn.

Cách này luôn cho **một** tập tổng `T`, `K` phần tử. Nhưng có phải tập nhỏ nhất theo từ điển không? Thử ngay:

In [18]:
def reconstruct_backward_prefix(n, s):
    """Truy vết 'kiểu sách giáo khoa' từ phải sang trái trên bảng TIỀN TỐ (không đảm bảo thứ tự từ điển)."""
    primes = sieve_primes(n)
    m = len(primes)
    pre = [[INF] * (s + 1) for _ in range(m + 1)]   # pre[i][t]: chỉ dùng primes[:i]
    pre[0][0] = 0
    for i in range(1, m + 1):
        p = primes[i - 1]
        for t in range(s + 1):
            pre[i][t] = pre[i - 1][t]
            if t >= p and pre[i - 1][t - p] + 1 < pre[i][t]:
                pre[i][t] = pre[i - 1][t - p] + 1
    T = max((t for t in range(1, s + 1) if pre[m][t] < INF), default=0)
    ans, rem, i = [], T, m
    while rem > 0:
        if pre[i][rem] == pre[i - 1][rem]:     # bỏ được primes[i-1] mà vẫn tối ưu -> bỏ
            i -= 1
        else:                                  # bắt buộc chọn primes[i-1]
            ans.append(primes[i - 1])
            rem -= primes[i - 1]
            i -= 1
    return sorted(ans)


print("Truy vết ngược trên tiền tố:", reconstruct_backward_prefix(15, 18))
print("Đáp án đúng (brute force)  :", prime_subset_bruteforce(15, 18))

Truy vết ngược trên tiền tố: [7, 11]
Đáp án đúng (brute force)  : [5, 13]


Truy vết ngược ra `[7, 11]` — **đúng tổng, đúng số phần tử, nhưng sai thứ tự từ điển**!

**Vì sao?** Đi từ phải sang trái nghĩa là ta quyết định **phần tử lớn nhất (cuối mảng) trước**: thấy bỏ được `13` thì bỏ. Nhưng thứ tự từ điển lại do **phần tử đầu mảng** quyết định trước tiên. Ta đã tối ưu sai đầu.

## 7.3 Ý tưởng đột phá: quyết định **từ trái sang phải**, hỏi "nhà tiên tri" về **hậu tố**

Muốn mảng nhỏ nhất theo từ điển, ta muốn **phần tử đầu tiên nhỏ nhất có thể**, rồi phần tử thứ hai nhỏ nhất có thể, ... Vậy hãy xét `primes` theo thứ tự **tăng dần** và tại mỗi số `p_i` hỏi:

> "Nếu tôi chọn `p_i` ngay bây giờ, thì phần tổng còn lại `rem − p_i` có tạo được bằng **các số đứng sau** `p_i` với **đúng số lượng tối ưu còn lại** không?"

Để trả lời trong `O(1)`, ta cần bảng DP về **hậu tố** (suffix — phần đuôi của mảng):

> `suf[i][t]` = số phần tử **ít nhất** chọn từ `primes[i:]` (các số từ chỉ số `i` trở đi) để có tổng **đúng** `t`; `INF` nếu không thể.

- **Cơ sở:** `suf[m][0] = 0` (không còn số nào, chỉ tạo được tổng 0 bằng 0 số), `suf[m][t] = INF` với `t > 0`.
- **Chuyển:** `suf[i][t] = min( suf[i+1][t],  1 + suf[i+1][t − p_i] )` — bỏ qua `p_i` hoặc chọn `p_i` (vế thứ hai chỉ khi `t ≥ p_i`).
- **Thứ tự tính:** `i` từ `m − 1` **về** `0` (hậu tố ngắn trước, dài sau).

Đây chính là khung "chọn / bỏ qua" ở mục 3.7, nhưng dùng `min` thay cho cộng. Vì bảng 2 chiều tách riêng từng `i`, ta **không phải lo** chuyện duyệt `t` xuôi hay ngược: mỗi hàng `i` chỉ đọc hàng `i + 1`.

### Tính tay bảng nhỏ: `primes = [2, 3, 5, 7]`, `t = 0..12`

| hàng | dùng được | t=0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| `suf[4]` | `[]` | 0 | ∞ | ∞ | ∞ | ∞ | ∞ | ∞ | ∞ | ∞ | ∞ | ∞ | ∞ | ∞ |
| `suf[3]` | `[7]` | 0 | ∞ | ∞ | ∞ | ∞ | ∞ | ∞ | 1 | ∞ | ∞ | ∞ | ∞ | ∞ |
| `suf[2]` | `[5, 7]` | 0 | ∞ | ∞ | ∞ | ∞ | 1 | ∞ | 1 | ∞ | ∞ | ∞ | ∞ | 2 |
| `suf[1]` | `[3, 5, 7]` | 0 | ∞ | ∞ | 1 | ∞ | 1 | ∞ | 1 | 2 | ∞ | 2 | ∞ | 2 |
| `suf[0]` | `[2, 3, 5, 7]` | 0 | ∞ | 1 | 1 | ∞ | 1 | ∞ | 1 | 2 | 2 | 2 | ∞ | 2 |

Ví dụ một ô: `suf[1][8] = min(suf[2][8], 1 + suf[2][8 − 3]) = min(∞, 1 + 1) = 2` (tập `[3, 5]`).

In [19]:
def build_suffix_table(primes, s):
    """suf[i][t] = số phần tử ít nhất chọn từ primes[i:] để có tổng ĐÚNG t (INF nếu không thể)."""
    m = len(primes)
    suf = [[INF] * (s + 1) for _ in range(m + 1)]
    suf[m][0] = 0                                   # hết số: chỉ tạo được tổng 0 bằng 0 phần tử
    for i in range(m - 1, -1, -1):                  # hậu tố ngắn -> dài
        p = primes[i]
        for t in range(s + 1):
            skip = suf[i + 1][t]                                 # không chọn primes[i]
            take = suf[i + 1][t - p] + 1 if t >= p else INF      # chọn primes[i]
            suf[i][t] = min(skip, take)
    return suf


small = [2, 3, 5, 7]
suf_small = build_suffix_table(small, 12)
print(f"{'hàng':<22}", ' '.join(f"{t:>2}" for t in range(13)))
for i in range(len(small), -1, -1):
    label = f"suf[{i}] {str(small[i:])}"
    print(f"{label:<22}", ' '.join(f"{fmt(x):>2}" for x in suf_small[i]))

hàng                    0  1  2  3  4  5  6  7  8  9 10 11 12
suf[4] []               0  ∞  ∞  ∞  ∞  ∞  ∞  ∞  ∞  ∞  ∞  ∞  ∞
suf[3] [7]              0  ∞  ∞  ∞  ∞  ∞  ∞  1  ∞  ∞  ∞  ∞  ∞
suf[2] [5, 7]           0  ∞  ∞  ∞  ∞  1  ∞  1  ∞  ∞  ∞  ∞  2
suf[1] [3, 5, 7]        0  ∞  ∞  1  ∞  1  ∞  1  2  ∞  2  ∞  2
suf[0] [2, 3, 5, 7]     0  ∞  1  1  ∞  1  ∞  1  2  2  2  ∞  2


Bảng in ra khớp từng ô với bảng tính tay ở trên.

## 7.4 Quy tắc dựng đáp án

Đặt `rem = T` (tổng còn phải gom). Duyệt `i = 0, 1, ..., m − 1`:

```text
nếu p_i ≤ rem  và  1 + suf[i+1][rem − p_i] == suf[i][rem]:
    chọn p_i:  ans.append(p_i);  rem -= p_i
ngược lại:
    bỏ qua p_i
dừng khi rem == 0
```

Diễn giải bằng lời:
- `suf[i][rem]` = số phần tử **còn cần** (tối ưu) nếu chỉ dùng các số từ `i` trở đi.
- Điều kiện `1 + suf[i+1][rem − p_i] == suf[i][rem]` nghĩa là: "chọn `p_i` **vẫn giữ được số lượng tối ưu**".
- Nếu chọn được thì **phải chọn**, vì mọi cách khác sẽ đặt ở vị trí này một số **lớn hơn** `p_i` → mảng lớn hơn theo từ điển.
- Nếu không chọn được thì `p_i` không thể nằm trong bất kỳ đáp án tối ưu nào (với phần đầu đã chọn) → bỏ qua an toàn.

Không cần biến `K` riêng: `suf[0][T]` chính là `K`, và mỗi lần chọn, `suf[i][rem]` giảm đúng 1.

In [20]:
def prime_subset_suffix_dp(n: int, s: int) -> List[int]:
    primes = sieve_primes(n)
    m = len(primes)
    suf = build_suffix_table(primes, s)

    # Bước A: tổng lớn nhất T (> 0) tạo được
    T = 0
    for t in range(s, 0, -1):
        if suf[0][t] < INF:
            T = t
            break
    if T == 0:
        return []                         # không có tập con khác rỗng hợp lệ

    # Bước B + C: đi từ trái sang phải, chọn số nhỏ nhất còn "khả thi tối ưu"
    ans = []
    rem = T
    for i in range(m):
        if rem == 0:
            break
        p = primes[i]
        if p <= rem and suf[i + 1][rem - p] + 1 == suf[i][rem]:
            ans.append(p)
            rem -= p
    return ans


for n, s, expected in examples:
    got = prime_subset_suffix_dp(n, s)
    print(f"n = {n:>2}, s = {s:>2} -> {got}  (mong đợi {expected})  {'OK' if got == expected else 'SAI'}")
print("n = 7, s = 8 ->", prime_subset_suffix_dp(7, 8), "| n = 1000, s = 1000 ->", prime_subset_suffix_dp(1000, 1000))

n =  7, s = 20 -> [2, 3, 5, 7]  (mong đợi [2, 3, 5, 7])  OK
n = 15, s = 18 -> [5, 13]  (mong đợi [5, 13])  OK
n =  5, s =  1 -> []  (mong đợi [])  OK
n = 7, s = 8 -> [3, 5] | n = 1000, s = 1000 -> [3, 997]


## 7.5 Góc nhìn đệ quy (top-down) của cùng thuật toán

Bảng `suf` chính là hàm đệ quy `f(i, t)` được **ghi nhớ** (mục 3.7):

```text
f(i, t) = 0                                     nếu t == 0
        = INF                                   nếu i == m (và t > 0)
        = min(f(i+1, t), 1 + f(i+1, t − p_i))   ngược lại
```

Viết theo kiểu này thường **dễ nghĩ ra hơn** khi mới gặp bài; viết theo kiểu bảng (bottom-up) thì **nhanh hơn và không lo giới hạn đệ quy**. Độ sâu đệ quy ở đây tối đa `m + 1 ≤ 169`, an toàn với giới hạn mặc định 1000 của Python.

In [21]:
def prime_subset_memo(n: int, s: int) -> List[int]:
    primes = sieve_primes(n)
    m = len(primes)

    @lru_cache(maxsize=None)
    def f(i, t):
        if t == 0:
            return 0
        if i == m:
            return INF
        res = f(i + 1, t)                              # bỏ qua primes[i]
        if primes[i] <= t:
            res = min(res, 1 + f(i + 1, t - primes[i]))  # chọn primes[i]
        return res

    T = 0
    for t in range(s, 0, -1):
        if f(0, t) < INF:
            T = t
            break
    ans, rem = [], T
    for i in range(m):
        if rem == 0:
            break
        p = primes[i]
        if p <= rem and 1 + f(i + 1, rem - p) == f(i, rem):
            ans.append(p)
            rem -= p
    return ans


for n, s, expected in examples:
    print(f"n = {n:>2}, s = {s:>2} -> {prime_subset_memo(n, s)}  (mong đợi {expected})")

n =  7, s = 20 -> [2, 3, 5, 7]  (mong đợi [2, 3, 5, 7])
n = 15, s = 18 -> [5, 13]  (mong đợi [5, 13])
n =  5, s =  1 -> []  (mong đợi [])


## 7.6 Độ phức tạp Cách 3

- Sàng: `O(n log log n)`.
- Bảng `suf`: `(m + 1) × (s + 1)` ô, mỗi ô `O(1)` → **Time `O(m · s)`**, **Space `O(m · s)`** (khoảng `169 × 1001 ≈ 1.7 · 10^5` số).
- Tìm `T`: `O(s)`; dựng đáp án: `O(m)`.

So với Cách 2, ta bỏ được thừa số `K` và chỉ lưu số nguyên. Đổi lại phải giữ cả bảng 2 chiều (cần để "hỏi tiên tri" ở mọi `i`).

---

# 8. Mô phỏng (Trace) bước dựng đáp án của Cách 3

Ví dụ 2: `n = 15, s = 18`, `primes = [2, 3, 5, 7, 11, 13]`, `T = 18`, `K = suf[0][18] = 2`.

| i | `p_i` | `rem` trước | Cần `suf[i][rem]` | Nếu chọn: `1 + suf[i+1][rem − p_i]` | Quyết định | `ans` sau |
|---:|---:|---:|---|---|---|---|
| 0 | 2 | 18 | `suf[0][18] = 2` | `1 + suf[1][16] = 1 + 2 = 3` | bỏ qua (3 ≠ 2) | `[]` |
| 1 | 3 | 18 | `suf[1][18] = 2` | `1 + suf[2][15] = 1 + ∞ = ∞` | bỏ qua | `[]` |
| 2 | 5 | 18 | `suf[2][18] = 2` | `1 + suf[3][13] = 1 + 1 = 2` | **chọn** | `[5]` |
| 3 | 7 | 13 | `suf[3][13] = 1` | `1 + suf[4][6] = ∞` | bỏ qua | `[5]` |
| 4 | 11 | 13 | `suf[4][13] = 1` | `1 + suf[5][2] = ∞` | bỏ qua | `[5]` |
| 5 | 13 | 13 | `suf[5][13] = 1` | `1 + suf[6][0] = 1 + 0 = 1` | **chọn** | `[5, 13]` |

Ý nghĩa từng cột:
- Hàng `i = 0`: chọn `2` thì phần còn lại `16` cần 2 số nữa (`3 + 13` hoặc `5 + 11`) → tổng 3 số, **không tối ưu** → bỏ `2`. Đây là lý do `[2, 3, 13]` bị loại.
- Hàng `i = 1`: chọn `3` thì `15` không tạo được từ `{5, 7, 11, 13}` → bỏ.
- Hàng `i = 2`: chọn `5` thì `13` tạo được bằng 1 số (`13`) → tổng 2 số = tối ưu → **chọn ngay**, nên `[7, 11]` không bao giờ được xét.

In [22]:
def trace_reconstruct(n, s):
    primes = sieve_primes(n)
    m = len(primes)
    suf = build_suffix_table(primes, s)
    T = 0
    for t in range(s, 0, -1):
        if suf[0][t] < INF:
            T = t
            break
    print(f"n = {n}, s = {s}, primes = {primes}, T = {T}, K = {suf[0][T] if T else 0}")
    ans, rem = [], T
    for i in range(m):
        if rem == 0:
            print("rem = 0 -> dừng")
            break
        p = primes[i]
        need = suf[i][rem]
        if p <= rem:
            take = suf[i + 1][rem - p] + 1
            take_txt = f"1 + suf[{i + 1}][{rem - p}] = {fmt(take)}"
        else:
            take = INF
            take_txt = f"p = {p} > rem"
        decision = "CHỌN" if take == need else "bỏ qua"
        print(f"i = {i} | p = {p:>2} | rem = {rem:>2} | cần suf[{i}][{rem}] = {fmt(need)} | nếu chọn: {take_txt:<20} -> {decision}")
        if take == need:
            ans.append(p)
            rem -= p
    print("Đáp án:", ans)
    return ans


trace_reconstruct(15, 18)
print()
trace_reconstruct(7, 8)

n = 15, s = 18, primes = [2, 3, 5, 7, 11, 13], T = 18, K = 2
i = 0 | p =  2 | rem = 18 | cần suf[0][18] = 2 | nếu chọn: 1 + suf[1][16] = 3   -> bỏ qua
i = 1 | p =  3 | rem = 18 | cần suf[1][18] = 2 | nếu chọn: 1 + suf[2][15] = ∞   -> bỏ qua
i = 2 | p =  5 | rem = 18 | cần suf[2][18] = 2 | nếu chọn: 1 + suf[3][13] = 2   -> CHỌN
i = 3 | p =  7 | rem = 13 | cần suf[3][13] = 1 | nếu chọn: 1 + suf[4][6] = ∞    -> bỏ qua
i = 4 | p = 11 | rem = 13 | cần suf[4][13] = 1 | nếu chọn: 1 + suf[5][2] = ∞    -> bỏ qua
i = 5 | p = 13 | rem = 13 | cần suf[5][13] = 1 | nếu chọn: 1 + suf[6][0] = 1    -> CHỌN
Đáp án: [5, 13]

n = 7, s = 8, primes = [2, 3, 5, 7], T = 8, K = 2
i = 0 | p =  2 | rem =  8 | cần suf[0][8] = 2 | nếu chọn: 1 + suf[1][6] = ∞    -> bỏ qua
i = 1 | p =  3 | rem =  8 | cần suf[1][8] = 2 | nếu chọn: 1 + suf[2][5] = 2    -> CHỌN
i = 2 | p =  5 | rem =  5 | cần suf[2][5] = 1 | nếu chọn: 1 + suf[3][0] = 1    -> CHỌN
rem = 0 -> dừng
Đáp án: [3, 5]


[3, 5]

Trace thứ hai (`n = 7, s = 8`) cho thấy một nhánh đáng chú ý: ở `i = 0`, chọn `2` thì `6` **không tạo được** từ `{3, 5, 7}` nên `2` bị bỏ, dù `2` là số nhỏ nhất. Đáp án `[3, 5]` khớp với tính tay ở Ví dụ A. Vòng lặp dừng ngay khi `rem = 0`, không cần xét `7`.

---

# 9. Phiên bản nộp LeetCode

Chuyển Cách 3 vào `class Solution`. Code nộp bài nên **gọn, không print**, tự chứa mọi bước (sàng, bảng, dựng đáp án). Hai tối ưu nhỏ cho Python:
- Gán `cur, nxt = suf[i], suf[i + 1]` để tránh tra chỉ số hai lần trong vòng lặp trong.
- Dùng `if ... < best` thay vì gọi `min(...)` (gọi hàm trong Python tốn hơn so sánh trực tiếp).

Nhắc lại: tên phương thức `primeSubset` là **giả định** (trang đề không có skeleton) — khi nộp hãy dùng đúng tên trong khung LeetCode.

In [23]:
from typing import List


class Solution:
    def primeSubset(self, n: int, s: int) -> List[int]:
        # 1) Sàng Eratosthenes: các số nguyên tố <= n, tăng dần
        if n < 2:
            return []
        is_prime = [True] * (n + 1)
        is_prime[0] = is_prime[1] = False
        i = 2
        while i * i <= n:
            if is_prime[i]:
                for j in range(i * i, n + 1, i):
                    is_prime[j] = False
            i += 1
        primes = [x for x in range(2, n + 1) if is_prime[x]]
        m = len(primes)

        # 2) suf[i][t] = số phần tử ít nhất từ primes[i:] để có tổng đúng t
        INF = float('inf')
        suf = [[INF] * (s + 1) for _ in range(m + 1)]
        suf[m][0] = 0
        for i in range(m - 1, -1, -1):
            p = primes[i]
            cur, nxt = suf[i], suf[i + 1]
            for t in range(s + 1):
                best = nxt[t]
                if t >= p and nxt[t - p] + 1 < best:
                    best = nxt[t - p] + 1
                cur[t] = best

        # 3) Tổng lớn nhất T > 0 tạo được (T = 0 -> trả về [])
        T = 0
        for t in range(s, 0, -1):
            if suf[0][t] < INF:
                T = t
                break

        # 4) Dựng đáp án nhỏ nhất theo từ điển, từ trái sang phải
        ans = []
        rem = T
        for i in range(m):
            if rem == 0:
                break
            p = primes[i]
            if p <= rem and suf[i + 1][rem - p] + 1 == suf[i][rem]:
                ans.append(p)
                rem -= p
        return ans

In [24]:
sol = Solution()

# Ví dụ trong đề
assert sol.primeSubset(7, 20) == [2, 3, 5, 7]
assert sol.primeSubset(15, 18) == [5, 13]
assert sol.primeSubset(5, 1) == []

# Edge cases
assert sol.primeSubset(1, 1000) == []          # không có số nguyên tố nào <= 1
assert sol.primeSubset(2, 1) == []             # có số 2 nhưng s quá nhỏ
assert sol.primeSubset(2, 2) == [2]            # tập 1 phần tử nhỏ nhất
assert sol.primeSubset(1000, 2) == [2]
assert sol.primeSubset(3, 4) == [3]            # 4 không tạo được (2+2 bị cấm)
assert sol.primeSubset(5, 6) == [5]            # 6 không tạo được; [5] thắng [2, 3]
assert sol.primeSubset(7, 8) == [3, 5]         # tham lam lớn-trước sai ở đây
assert sol.primeSubset(4, 1000) == [2, 3]      # lấy hết mà vẫn < s
assert sol.primeSubset(1000, 4) == [3]
assert sol.primeSubset(1000, 6) == [5]
assert sol.primeSubset(1000, 27) == [3, 5, 19] # 27 lẻ, không phải nguyên tố -> cần 3 số
assert sol.primeSubset(1000, 1000) == [3, 997] # input lớn nhất
assert sol.primeSubset(89, 963) == sieve_primes(89) and len(sieve_primes(89)) == 24  # đáp án dài nhất có thể

print("Tất cả test của class Solution: PASS")
print("n = 1000, s = 1000 ->", sol.primeSubset(1000, 1000))
print("n = 89, s = 963 ->", sol.primeSubset(89, 963))

Tất cả test của class Solution: PASS
n = 1000, s = 1000 -> [3, 997]
n = 89, s = 963 -> [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89]


# 10. Kiểm tra chéo (cross-check) với brute force

Một lời giải Hard "chạy đúng 3 ví dụ" là **chưa đủ**. Ta sinh ngẫu nhiên nhiều input nhỏ (có `seed` cố định để chạy lại ra cùng kết quả) và so **mọi cách** với brute force (trọng tài hiển nhiên đúng).

- `n ≤ 40` (tối đa 12 số nguyên tố → 4096 tập con) để brute force chạy nhanh.
- `s ≤ 200` để có cả trường hợp lấy hết, lấy một phần, và `s` quá nhỏ.
- Thêm một vòng so Cách 2 với Cách 3 trên input **lớn** (`n, s ≤ 1000`), nơi brute force không chạy nổi.

In [25]:
import random

rng = random.Random(4081)
approaches = [prime_subset_dp_lists, prime_subset_suffix_dp, prime_subset_memo, Solution().primeSubset]

small_cases = 0
for _ in range(250):
    n = rng.randint(1, 40)
    s = rng.randint(1, 200)
    expected = prime_subset_bruteforce(n, s)
    for f in approaches:
        got = f(n, s)
        assert got == expected, (f.__name__, n, s, got, expected)
    small_cases += 1

large_cases = 0
for _ in range(60):
    n = rng.randint(1, 1000)
    s = rng.randint(1, 1000)
    a = prime_subset_dp_lists(n, s)
    b = Solution().primeSubset(n, s)
    assert a == b, (n, s, a, b)
    large_cases += 1

print(f"{small_cases} input nhỏ: 4 cách đều khớp brute force")
print(f"{large_cases} input lớn: Cách 2 khớp Cách 3 (class Solution)")

250 input nhỏ: 4 cách đều khớp brute force
60 input lớn: Cách 2 khớp Cách 3 (class Solution)


# 11. Vì sao thuật toán đúng?

## Cách 2 — bất biến (invariant) của `best`

> Sau khi đã thêm `p_1 < p_2 < ... < p_j`, với mọi `t`, `best[t]` là tập tốt nhất (ít phần tử nhất; hòa thì nhỏ nhất theo từ điển) trong các tập con của `{p_1, ..., p_j}` có tổng đúng `t`.

- **Ban đầu** (`j = 0`): chỉ có tập rỗng, tổng 0 → `best[0] = ()`, còn lại `None`. Đúng.
- **Thêm `p = p_{j+1}`:** tập mới tổng `t` hoặc không chứa `p` (tốt nhất là `best[t]` cũ), hoặc là `X + [p]` với `X ⊆ {p_1..p_j}` tổng `t − p`. Vì `p` lớn nhất nên luôn đứng cuối, việc so sánh `X + [p]` với `Y + [p]` **tương đương** so `X` với `Y` (cùng tiêu chí số lượng rồi từ điển). Nên ứng viên tốt nhất chứa `p` là `best[t − p]` cũ + `(p,)`. Lấy cái tốt hơn trong hai → bất biến giữ nguyên.
- **Duyệt `t` ngược** bảo đảm `best[t − p]` lúc đọc vẫn là giá trị **cũ** (chưa chứa `p`).
- **Kết thúc:** tiêu chí 1 chọn `T` lớn nhất có `best[T] ≠ None`; `best[T]` thỏa tiêu chí 2 và 3 theo bất biến.

## Cách 3 — bảng hậu tố + chọn tham lam có kiểm tra

1. **Bảng đúng:** mọi tập con của `primes[i:]` hoặc không chứa `p_i` (thuộc `primes[i+1:]`) hoặc chứa `p_i`. Công thức `min(bỏ, 1 + chọn)` xét đủ hai trường hợp → `suf[i][t]` đúng bằng số phần tử ít nhất.
2. **Bất biến khi dựng:** gọi `A*` là đáp án đúng (tập tổng `T`, `K` phần tử, nhỏ nhất theo từ điển). Trước khi xét chỉ số `i`: `ans` là **phần đầu** của `A*`, các phần tử còn lại của `A*` đều nằm trong `primes[i:]`, `rem = T − sum(ans)` và `len(ans) + suf[i][rem] = K`.
3. **Bước chọn:** nếu `1 + suf[i+1][rem − p_i] == suf[i][rem]` thì tồn tại một cách hoàn thành tối ưu có phần tử kế tiếp là `p_i`. Phần tử kế tiếp `q` của `A*` nằm trong `primes[i:]` nên `q ≥ p_i`. Nếu `q > p_i` thì cách hoàn thành bắt đầu bằng `p_i` cho một tập **nhỏ hơn** `A*` theo từ điển — mâu thuẫn. Vậy `q = p_i`: chọn `p_i` là đúng, bất biến giữ với `i + 1`.
4. **Bước bỏ:** nếu điều kiện sai, không cách hoàn thành tối ưu nào có phần tử kế tiếp là `p_i`, nên phần tử kế tiếp của `A*` nằm trong `primes[i+1:]`; đồng thời `min` đạt ở nhánh "bỏ" nên `suf[i+1][rem] = suf[i][rem]` → bất biến giữ với `i + 1`.
5. **Dừng** khi `rem = 0`: `ans` có tổng `T`, đúng `K` phần tử, và từng vị trí đều nhỏ nhất có thể → nhỏ nhất theo từ điển.

Mấu chốt cần nhớ: **muốn quyết định từ trái sang phải thì bảng phải mô tả phần bên phải (hậu tố)**.

## Độ phức tạp và so sánh các cách

Ký hiệu: `n` = giới hạn số nguyên tố, `m` = số số nguyên tố `≤ n` (`m ≤ 168`), `s` = giới hạn tổng (`≤ 1000`), `K` = độ dài đáp án (`≤ 24`). Mọi cách đều cộng thêm `O(n log log n)` cho sàng.

| Cách | Ý tưởng | Time | Space | Dùng khi nào |
|---|---|---|---|---|
| 1. Brute force | thử `2^m` tập con, khóa `(-tổng, số lượng, mảng)` | `O(2^m · m)` | `O(m)` | trọng tài cho `n ≤ ~50` |
| 2. DP lưu tuple | `best[t]` = tập tốt nhất tổng đúng `t` | `O(m · s · K)` | `O(s · K)` | đáp án ngắn, cần code nhanh |
| 3. DP hậu tố + tham lam | `suf[i][t]` = số lượng ít nhất; chọn từ trái | `O(m · s)` | `O(m · s)` | **chuẩn**, tổng quát, dễ chuyển sang C++/Java |
| 3'. Top-down memo | cùng công thức với `lru_cache` | `O(m · s)` | `O(m · s)` + stack | dễ nghĩ, chậm hơn một chút |

Python thuần chạy Cách 2 và Cách 3 rất nhanh ở input lớn nhất. Đo thử (thời gian thay đổi theo máy, chỉ để tham khảo):

In [26]:
import time

for name, f in [("Cách 2 (DP tuple)", prime_subset_dp_lists),
                ("Cách 3 (DP hậu tố)", prime_subset_suffix_dp),
                ("Solution.primeSubset", Solution().primeSubset)]:
    start = time.perf_counter()
    for s in (997, 998, 999, 1000):
        f(1000, s)
    avg_ms = (time.perf_counter() - start) * 1000 / 4
    print(f"{name:<22}: trung bình {avg_ms:6.1f} ms mỗi lần gọi với n = 1000")

Cách 2 (DP tuple)     : trung bình    9.9 ms mỗi lần gọi với n = 1000
Cách 3 (DP hậu tố)    : trung bình   12.8 ms mỗi lần gọi với n = 1000
Solution.primeSubset  : trung bình    7.6 ms mỗi lần gọi với n = 1000


**Đọc kết quả đo thế nào?** Big-O mô tả tốc độ **tăng** khi input lớn lên, không mô tả hằng số. Ở giới hạn `s ≤ 1000`:

- Vòng trong của Cách 2 chỉ chạy `t` từ `s` về `p` (với `p` gần 1000 thì rất ngắn) và tuple chỉ dài vài phần tử, nên Cách 2 chạy khá nhanh.
- Bản Cách 3 "giảng dạy" duyệt đủ `t = 0..s` cho mọi `i` và gọi `min(...)` ở mỗi ô, nên trên máy của bạn nó **có thể không nhanh hơn** Cách 2 — điều đó hoàn toàn bình thường.
- Bản `Solution` cùng thuật toán Cách 3 nhưng tối ưu hằng số (biến cục bộ `cur, nxt`, so sánh trực tiếp thay cho `min`) thường nhanh nhất. Có thể tối ưu thêm: với `t < p` thì `suf[i][t] = suf[i+1][t]`, nên chép nguyên đoạn `cur[:p] = nxt[:p]` rồi chỉ lặp `t ≥ p`.

Cả ba đều chỉ tốn cỡ vài chục mili giây ở input lớn nhất → **Python thuần không có vấn đề gì về thời gian** ở bài này.

# 12. Những lỗi thường gặp

## Lỗi 1: Duyệt tổng **xuôi** trong 0/1 knapsack → dùng một số nhiều lần

Sai:

```python
for p in primes:
    for t in range(p, s + 1):      # XUÔI
        ...best[t] từ best[t - p]...
```

Khi duyệt xuôi, `best[t − p]` có thể **vừa được cập nhật bằng chính `p`** ở vòng này → `p` bị dùng 2 lần, 3 lần... Kết quả có thể chứa số lặp như `[2, 2]` — vi phạm "mỗi số tối đa một lần".

Sửa: `for t in range(s, p - 1, -1)` (duyệt **ngược**).

In [27]:
def buggy_forward(n, s):
    primes = sieve_primes(n)
    best = [None] * (s + 1)
    best[0] = ()
    for p in primes:
        for t in range(p, s + 1):                  # LỖI: duyệt xuôi
            prev = best[t - p]
            if prev is None:
                continue
            cand = prev + (p,)
            cur = best[t]
            if cur is None or (len(cand), cand) < (len(cur), cur):
                best[t] = cand
    for t in range(s, 0, -1):
        if best[t] is not None:
            return list(best[t])
    return []


for n, s in [(3, 4), (7, 20)]:
    print(f"n = {n}, s = {s}: duyệt xuôi -> {buggy_forward(n, s)} | đúng -> {prime_subset_dp_lists(n, s)}")

n = 3, s = 4: duyệt xuôi -> [2, 2] | đúng -> [3]
n = 7, s = 20: duyệt xuôi -> [3, 3, 7, 7] | đúng -> [2, 3, 5, 7]


## Lỗi 2: So sánh từ điển mà **quên so số phần tử trước**

Sai: `key = (-tổng, mảng)` hoặc trong Cách 2 viết `if cand < cur`.

Vì `[2, 3, 13] < [5, 13]` (mục 3.3), khóa thiếu `len` sẽ chọn tập 3 phần tử. Sửa: `key = (-tổng, len(mảng), mảng)`.

## Lỗi 3: So sánh dưới dạng **chuỗi** thay vì list số

Sai: nối mảng thành chuỗi rồi so (`"1113"` vs `"519"`), hoặc sort theo `str`. So sánh chuỗi so **từng ký tự**: `"1" < "5"` nên `"1113" < "519"`, trong khi số `11 > 5`. Sửa: luôn so sánh **list/tuple số nguyên**.

In [28]:
cands_18 = [[2, 3, 13], [7, 11], [5, 13], [2, 5, 11]]
print("Lỗi 2 - khóa thiếu len:", min(cands_18, key=lambda c: (-sum(c), c)), "| đúng:", min(cands_18, key=lambda c: (-sum(c), len(c), c)))

cands_24 = [[11, 13], [5, 19]]          # cùng tổng 24, cùng 2 phần tử
print("Lỗi 3 - so chuỗi:", min(cands_24, key=lambda c: ''.join(map(str, c))), "| so list số (đúng):", min(cands_24))
print("Kiểm chứng bằng brute force n = 19, s = 24:", prime_subset_bruteforce(19, 24))

Lỗi 2 - khóa thiếu len: [2, 3, 13] | đúng: [5, 13]
Lỗi 3 - so chuỗi: [11, 13] | so list số (đúng): [5, 19]
Kiểm chứng bằng brute force n = 19, s = 24: [5, 19]


## Lỗi 4: Sàng sai biên — bỏ mất `n` hoặc tính `1` là số nguyên tố

Sai thường gặp:
- `range(2, n)` thay vì `range(2, n + 1)` → nếu `n` là số nguyên tố (ví dụ `n = 7`) thì **mất số 7**; Ví dụ 1 sẽ ra `[2, 3, 5]`.
- Bắt đầu từ `1` và quên loại `1` → `1` lọt vào danh sách; Ví dụ 3 (`s = 1`) sẽ trả `[1]` thay vì `[]`.

Sửa: mảng `is_prime` kích thước `n + 1`, đặt `is_prime[0] = is_prime[1] = False`.

In [29]:
def bad_primes(n):
    # LỖI KÉP: range(1, n) vừa chứa 1, vừa bỏ mất n
    return [x for x in range(1, n) if all(x % d != 0 for d in range(2, x))]


print("bad_primes(7) =", bad_primes(7), "| sieve_primes(7) =", sieve_primes(7))
bad_sum = sum(bad_primes(7))
print("Ví dụ 1 với danh sách sai: tổng tất cả =", bad_sum, "(thiếu 7, thừa 1) -> sai đáp án")
print("Ví dụ 3 (s = 1) với danh sách sai sẽ chọn được", [x for x in bad_primes(5) if x <= 1], "thay vì []")

bad_primes(7) = [1, 2, 3, 5] | sieve_primes(7) = [2, 3, 5, 7]
Ví dụ 1 với danh sách sai: tổng tất cả = 11 (thiếu 7, thừa 1) -> sai đáp án
Ví dụ 3 (s = 1) với danh sách sai sẽ chọn được [1] thay vì []


## Lỗi 5: Tham lam (lớn trước / nhỏ trước) — đã thấy ở mục 5.3

`n = 7, s = 8`: lớn-trước ra `[7]`, đúng là `[3, 5]`. Không có quy tắc cục bộ nào thay được DP.

## Lỗi 6: Truy vết ngược trên bảng tiền tố — đã thấy ở mục 7.2

Ra `[7, 11]` thay vì `[5, 13]`: đúng tổng, đúng số lượng nhưng **sai thứ tự từ điển**. Muốn chọn từ trái thì phải dùng bảng **hậu tố**.

## Lỗi 7: Khởi tạo sai bảng hậu tố

- Quên `suf[m][0] = 0` → mọi ô thành `INF`, không tạo được tổng nào.
- Khởi tạo bằng `0` thay vì `INF` → `min` luôn ra 0, "tạo được" cả những tổng không thể.

## Lỗi 8: Xử lý sai trường hợp không có đáp án

Khi `T = 0` (ví dụ `n = 1` hoặc `s = 1`) phải trả `[]`, không phải `[0]`, `None`, hay `[1]`. Vòng tìm `T` nên chạy `t = s, s−1, ..., 1` (không tính `t = 0`, vì tập rỗng tổng 0 không phải "tập con khác rỗng").

## Bảng edge case nên test

| Input | Đáp án | Điểm cần chú ý |
|---|---|---|
| `n = 1, s = 1000` | `[]` | không có số nguyên tố |
| `n = 2, s = 1` | `[]` | có số nhưng `s` quá nhỏ |
| `n = 3, s = 4` | `[3]` | `4 = 2 + 2` bị cấm |
| `n = 5, s = 6` | `[5]` | `T < s`; ít phần tử thắng từ điển |
| `n = 4, s = 1000` | `[2, 3]` | lấy hết vẫn chưa tới `s` |
| `n = 1000, s = 27` | `[3, 5, 19]` | cần 3 số |
| `n = 89, s = 963` | 24 số nguyên tố `≤ 89` | đáp án dài nhất |

# 13. Mẹo nhận dạng (dấu hiệu nhận dạng) và bài liên quan

## Dấu hiệu nhận dạng

| Dấu hiệu trong đề | Nghĩ tới |
|---|---|
| "chọn tập con", "tổng `≤ s`" hoặc "tổng `= target`", và **giới hạn tổng nhỏ** (`≤ 10^3 – 10^4`) | DP theo tổng (subset sum / knapsack) — trạng thái là **tổng**, không phải tập |
| "mỗi phần tử dùng tối đa một lần" | 0/1 knapsack → duyệt tổng **ngược** (hoặc bảng 2 chiều theo chỉ số) |
| Nhiều tiêu chí "lớn nhất, rồi ít nhất, rồi nhỏ nhất..." | **Khóa tuple**; xử lý tiêu chí theo thứ tự ưu tiên |
| "return the lexicographically smallest" trên kết quả DP | DP trên **hậu tố** + chọn **tham lam từ trái** với kiểm tra khả thi |
| "các số nguyên tố `≤ n`" | **Sàng Eratosthenes** |
| Số tập con `2^m` quá lớn nhưng tổng nhỏ | Đổi "liệt kê tập" thành "liệt kê tổng" (pseudo-polynomial) |

## Bài liên quan để luyện tiếp

| Bài | Liên hệ |
|---|---|
| 204. Count Primes | Sàng Eratosthenes thuần |
| 2761. Prime Pairs With Target Sum | Sàng + tìm cặp số nguyên tố có tổng `n` |
| 416. Partition Equal Subset Sum | Subset sum (0/1), có thể dùng bitset |
| 1049. Last Stone Weight II | Subset sum: tổng lớn nhất `≤` một nửa |
| 322. Coin Change | Knapsack **không giới hạn** số lần, "ít phần tử nhất" (duyệt xuôi) |
| 474. Ones and Zeroes | 0/1 knapsack hai chiều tài nguyên |
| 1449. Form Largest Integer With Digits That Add up to Target | DP + **dựng đáp án tối ưu theo thứ tự** từ trái sang phải — cùng tư duy với Cách 3 |

Mini code: tái sử dụng hai kỹ thuật vừa học cho hai bài liên quan.

In [30]:
def can_partition(nums):
    """LeetCode 416 bằng bitset: có chia được thành 2 phần tổng bằng nhau không?"""
    total = sum(nums)
    if total % 2:
        return False
    bits = 1
    for x in nums:
        bits |= bits << x            # 0/1 knapsack bằng bitset
    return (bits >> (total // 2)) & 1 == 1


def prime_pairs(n):
    """LeetCode 2761: mọi cặp [x, y], x <= y, x + y == n, cả hai nguyên tố."""
    primes = sieve_primes(n)
    pset = set(primes)
    return [[x, n - x] for x in primes if x <= n - x and (n - x) in pset]


print(can_partition([1, 5, 11, 5]), can_partition([1, 2, 3, 5]))
print(prime_pairs(10), prime_pairs(2))

True False
[[3, 7], [5, 5]] []


# 14. Bài tập vận dụng (từ dễ đến khó)

Tự làm trước khi xem **Đáp án tham khảo** ở cuối notebook. Mỗi bài có **Gợi ý 3 mức**: (1) hướng tư duy → (2) cấu trúc/cú pháp → (3) gần-code. Chỉ mở mức tiếp theo khi thật sự bí.

## Bài 1 — Đoán output (so sánh list/tuple)

Không chạy code, hãy đoán từng dòng in ra gì:

```python
print([5, 13] < [7, 11])
print([2, 3, 13] < [5, 13])
print([11, 13] < [5, 19])
print("1113" < "519")
print((2, [5, 13]) < (3, [2, 3, 13]))
print((-18, 2, [5, 13]) < (-17, 1, [17]))
print(min([[7, 11], [5, 13], [2, 3, 13]], key=lambda a: (len(a), a)))
```

- **Gợi ý 1:** so phần tử đầu tiên; chỉ khi bằng nhau mới nhìn tiếp.
- **Gợi ý 2:** chuỗi so từng **ký tự** (`"1"` với `"5"`), list số so từng **số**.
- **Gợi ý 3:** ở dòng 6, thành phần đầu là `-18` và `-17` — cái nào nhỏ hơn? Có cần nhìn tiếp không?

## Bài 2 — Tính tay

Với `n = 10`:

1. Liệt kê các số nguyên tố `≤ 10`.
2. Với `s = 16`: tổng `16` có tạo được không? Tổng lớn nhất `T` là bao nhiêu? Đáp án là gì?
3. Với `s = 13`: tương tự. Có mấy tập đạt tổng `T`? Chọn tập nào?

- **Gợi ý 1:** tổng của tất cả các số nguyên tố `≤ 10` là bao nhiêu? Bắt đầu thử từ `T = s` rồi giảm dần.
- **Gợi ý 2:** để ý tính chẵn lẻ: các số nguyên tố lẻ cộng lại, số lượng chẵn → tổng chẵn; muốn tổng lẻ khi không dùng `2` thì cần số lượng lẻ.
- **Gợi ý 3:** với `s = 13`, thử `13 = 2 + 11`? (11 > 10). `13 = 3 + 10`? ... Nếu `13` không được thì xét `12 = 5 + 7 = 2 + 3 + 7`.

In [31]:
# TODO: điền kết quả tính tay của bạn (không cần viết hàm)
my_primes_10 = None      # ví dụ: [..]
my_T_16 = None
my_answer_16 = None
my_T_13 = None
my_answer_13 = None

# Bỏ comment để tự kiểm tra sau khi điền:
# assert my_primes_10 == sieve_primes(10)
# assert my_answer_16 == Solution().primeSubset(10, 16) and my_T_16 == sum(my_answer_16)
# assert my_answer_13 == Solution().primeSubset(10, 13) and my_T_13 == sum(my_answer_13)

## Bài 3 — Điền TODO: sàng Eratosthenes

Hoàn thiện `exercise_sieve(n)` trả về list số nguyên tố `≤ n` tăng dần.

- **Gợi ý 1:** gạch bỏ các bội của mỗi số chưa bị gạch.
- **Gợi ý 2:** mảng `is_prime` kích thước `n + 1`; `0` và `1` là `False`; vòng ngoài chạy khi `i * i <= n`.
- **Gợi ý 3:** `for j in range(i * i, n + 1, i): is_prime[j] = False`; cuối cùng `return [x for x in range(n + 1) if is_prime[x]]`.

In [32]:
def exercise_sieve(n):
    # TODO 1: nếu n < 2 thì trả về []
    # TODO 2: tạo is_prime = [True] * (n + 1), đặt is_prime[0] = is_prime[1] = False
    # TODO 3: với i = 2, 3, ... khi i * i <= n: nếu is_prime[i], gạch các bội từ i * i
    # TODO 4: trả về các x có is_prime[x] == True
    pass


# Tự làm xong rồi mới mở test:
# assert exercise_sieve(1) == []
# assert exercise_sieve(7) == [2, 3, 5, 7]
# assert exercise_sieve(30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
# assert len(exercise_sieve(1000)) == 168

## Bài 4 — Điền TODO: dựng đáp án từ bảng hậu tố

Cho sẵn `primes` (tăng dần), bảng `suf` (từ `build_suffix_table`) và tổng `T`. Hãy viết vòng dựng đáp án nhỏ nhất theo từ điển.

- **Gợi ý 1:** đi từ trái sang phải; với mỗi số, hỏi "chọn nó có còn giữ được số lượng tối ưu không?".
- **Gợi ý 2:** "số lượng tối ưu còn lại" là `suf[i][rem]`; nếu chọn `primes[i]` thì số lượng là `1 + suf[i + 1][rem - primes[i]]`.
- **Gợi ý 3:** `if p <= rem and suf[i + 1][rem - p] + 1 == suf[i][rem]: ans.append(p); rem -= p`, dừng khi `rem == 0`.

In [33]:
def exercise_reconstruct(primes, suf, T):
    # TODO 1: ans = [], rem = T
    # TODO 2: duyệt i từ 0 đến len(primes) - 1, dừng sớm nếu rem == 0
    # TODO 3: nếu chọn primes[i] vẫn tối ưu thì chọn (append, trừ rem)
    # TODO 4: trả về ans
    pass


# Tự làm xong rồi mới mở test:
# pr = sieve_primes(15)
# tb = build_suffix_table(pr, 18)
# assert exercise_reconstruct(pr, tb, 18) == [5, 13]
# pr7 = sieve_primes(7)
# assert exercise_reconstruct(pr7, build_suffix_table(pr7, 8), 8) == [3, 5]

## Bài 5 — Biến thể: **nhiều** số nguyên tố nhất

Giữ nguyên tiêu chí 1 (tổng lớn nhất `≤ s`) và tiêu chí 3 (nhỏ nhất theo từ điển), nhưng đổi tiêu chí 2 thành: chọn tập chứa **nhiều** số nguyên tố **nhất**.

Ví dụ: `n = 15, s = 18` → các tập tổng 18 nhiều phần tử nhất là `[2, 3, 13]` và `[2, 5, 11]` (3 phần tử) → đáp án `[2, 3, 13]`.

- **Gợi ý 1:** Cách 3 vẫn dùng được; chỉ đổi "ít nhất" thành "nhiều nhất".
- **Gợi ý 2:** "không thể" lúc này phải là `-INF` (âm vô cực) để `max` không chọn nhầm; `suf[m][0] = 0`.
- **Gợi ý 3:** `suf[i][t] = max(suf[i+1][t], 1 + suf[i+1][t-p])`; điều kiện chọn khi dựng **giữ nguyên dạng** `1 + suf[i+1][rem-p] == suf[i][rem]`.

In [34]:
def exercise_most_primes(n, s):
    # TODO 1: primes = sieve_primes(n); NEG = float('-inf')
    # TODO 2: bảng suf kích thước (m + 1) x (s + 1), khởi tạo NEG, suf[m][0] = 0
    # TODO 3: suf[i][t] = max(bỏ, 1 + chọn)
    # TODO 4: T = tổng lớn nhất > 0 có suf[0][T] > NEG; nếu không có thì trả về []
    # TODO 5: dựng đáp án từ trái sang phải
    pass


# Tự làm xong rồi mới mở test:
# assert exercise_most_primes(15, 18) == [2, 3, 13]
# assert exercise_most_primes(7, 20) == [2, 3, 5, 7]
# assert exercise_most_primes(10, 16) == [3, 5, 7]
# assert exercise_most_primes(5, 1) == []

## Bài 6 — Mini challenge: tổng quát hóa cho mảng bất kỳ

Cho mảng `nums` gồm các số nguyên **dương** (có thể **trùng nhau**; mỗi **phần tử** dùng tối đa một lần) và số `s`. Áp dụng đúng 3 tiêu chí của đề (tổng lớn nhất `≤ s` → ít phần tử nhất → nhỏ nhất theo từ điển khi viết tăng dần). Trả về mảng đáp án.

Ví dụ:
- `nums = [6, 4, 4, 1, 3], s = 9` → `[3, 6]` (thắng `[1, 4, 4]` vì ít phần tử hơn).
- `nums = [5, 5, 2, 8], s = 10` → `[2, 8]` (cùng 2 phần tử với `[5, 5]`, nhưng `2 < 5`).
- `nums = [2, 2, 3, 7], s = 7` → `[7]`.

- **Gợi ý 1:** điều gì trong Cách 3 phụ thuộc vào việc các số là số nguyên tố? (Gần như không gì cả!)
- **Gợi ý 2:** điều kiện "mảng tăng dần" cần `nums` được **sắp xếp** trước; số trùng nhau không làm hỏng lập luận "chọn sớm nhất có thể".
- **Gợi ý 3:** `a = sorted(nums)`, rồi copy nguyên `build_suffix_table(a, s)` + vòng tìm `T` + vòng dựng đáp án.

In [35]:
def exercise_general_subset(nums, s):
    # TODO 1: sắp xếp nums tăng dần
    # TODO 2: dựng bảng hậu tố số phần tử ít nhất
    # TODO 3: tìm T, dựng đáp án từ trái sang phải
    pass


# Tự làm xong rồi mới mở test:
# assert exercise_general_subset([6, 4, 4, 1, 3], 9) == [3, 6]
# assert exercise_general_subset([5, 5, 2, 8], 10) == [2, 8]
# assert exercise_general_subset([2, 2, 3, 7], 7) == [7]
# assert exercise_general_subset([4, 4, 4], 3) == []

# Đáp án tham khảo

Hãy tự làm trước khi đọc phần này. Mỗi đáp án đều có `assert` để tự kiểm chứng.

## Đáp án Bài 1 (đoán output)

| Dòng | Kết quả | Lý do |
|---|---|---|
| `[5, 13] < [7, 11]` | `True` | `5 < 7` |
| `[2, 3, 13] < [5, 13]` | `True` | `2 < 5` — độ dài không được xét trước |
| `[11, 13] < [5, 19]` | `False` | `11 > 5` |
| `"1113" < "519"` | `True` | ký tự `"1" < "5"` — đây là lý do không so chuỗi |
| `(2, [5, 13]) < (3, [2, 3, 13])` | `True` | `2 < 3` quyết định |
| `(-18, 2, [5, 13]) < (-17, 1, [17])` | `True` | `-18 < -17`: tổng 18 tốt hơn tổng 17, không cần nhìn tiếp |
| `min(..., key=lambda a: (len(a), a))` | `[5, 13]` | 2 phần tử, rồi `5 < 7` |

In [36]:
# Đáp án Bài 1: chạy để đối chiếu
print([5, 13] < [7, 11])
print([2, 3, 13] < [5, 13])
print([11, 13] < [5, 19])
print("1113" < "519")
print((2, [5, 13]) < (3, [2, 3, 13]))
print((-18, 2, [5, 13]) < (-17, 1, [17]))
print(min([[7, 11], [5, 13], [2, 3, 13]], key=lambda a: (len(a), a)))

True
True
False
True
True
True
[5, 13]


## Đáp án Bài 2 (tính tay)

- Số nguyên tố `≤ 10`: `2, 3, 5, 7` (tổng 17).
- `s = 16`: muốn tổng 16 (chẵn). Nếu không dùng `2` thì cần số chẵn các số lẻ: `3+5=8`, `3+7=10`, `5+7=12` — không ra 16. Nếu dùng `2` thì cần `14` từ `{3,5,7}` với số lẻ phần tử: `3+5+7 = 15` — không ra. → `16` **không tạo được**. Thử `15 = 3 + 5 + 7` ✔ (và đây là cách duy nhất). Đáp án `[3, 5, 7]`, `T = 15`.
- `s = 13`: `13` không tạo được (kiểm tra tương tự). `12 = 5 + 7 = 2 + 3 + 7` → 2 tập; chọn `[5, 7]` (ít phần tử hơn), `T = 12`.

In [37]:
from itertools import combinations

answer_primes_10 = sieve_primes(10)
for target in (16, 15, 13, 12):
    subsets = [list(c) for r in range(1, len(answer_primes_10) + 1)
               for c in combinations(answer_primes_10, r) if sum(c) == target]
    print(f"tổng {target}: {subsets}")

assert answer_primes_10 == [2, 3, 5, 7]
assert Solution().primeSubset(10, 16) == [3, 5, 7]
assert Solution().primeSubset(10, 13) == [5, 7]
print("Bài 2: PASS")

tổng 16: []
tổng 15: [[3, 5, 7]]
tổng 13: []
tổng 12: [[5, 7], [2, 3, 7]]
Bài 2: PASS


## Đáp án Bài 3 và Bài 4

In [38]:
def answer_sieve(n):
    if n < 2:
        return []
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False
    i = 2
    while i * i <= n:
        if is_prime[i]:
            for j in range(i * i, n + 1, i):
                is_prime[j] = False
        i += 1
    return [x for x in range(n + 1) if is_prime[x]]


def answer_reconstruct(primes, suf, T):
    ans, rem = [], T
    for i in range(len(primes)):
        if rem == 0:
            break
        p = primes[i]
        if p <= rem and suf[i + 1][rem - p] + 1 == suf[i][rem]:
            ans.append(p)
            rem -= p
    return ans


assert answer_sieve(1) == []
assert answer_sieve(7) == [2, 3, 5, 7]
assert answer_sieve(30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert len(answer_sieve(1000)) == 168

pr = sieve_primes(15)
assert answer_reconstruct(pr, build_suffix_table(pr, 18), 18) == [5, 13]
pr7 = sieve_primes(7)
assert answer_reconstruct(pr7, build_suffix_table(pr7, 8), 8) == [3, 5]
print("Bài 3, Bài 4: PASS")

Bài 3, Bài 4: PASS


## Đáp án Bài 5 (nhiều số nguyên tố nhất)

Chỉ cần đổi `INF → -INF` và `min → max`. Lập luận "chọn sớm nhất có thể" vẫn đúng y nguyên: nếu chọn `p_i` vẫn đạt số lượng tối ưu (lớn nhất) thì phải chọn, vì mọi lựa chọn khác đặt số lớn hơn ở vị trí này. Ta kiểm tra chéo với một brute force dùng khóa `(-tổng, -số_lượng, mảng)`.

In [39]:
def answer_most_primes(n, s):
    primes = sieve_primes(n)
    m = len(primes)
    NEG = float('-inf')
    suf = [[NEG] * (s + 1) for _ in range(m + 1)]
    suf[m][0] = 0
    for i in range(m - 1, -1, -1):
        p = primes[i]
        for t in range(s + 1):
            best = suf[i + 1][t]
            if t >= p and suf[i + 1][t - p] + 1 > best:
                best = suf[i + 1][t - p] + 1
            suf[i][t] = best
    T = 0
    for t in range(s, 0, -1):
        if suf[0][t] > NEG:
            T = t
            break
    ans, rem = [], T
    for i in range(m):
        if rem == 0:
            break
        p = primes[i]
        if p <= rem and suf[i + 1][rem - p] + 1 == suf[i][rem]:
            ans.append(p)
            rem -= p
    return ans


def most_primes_bruteforce(n, s):
    primes = sieve_primes(n)
    m = len(primes)
    best_key, best = None, []
    for mask in range(1, 1 << m):
        c = [primes[i] for i in range(m) if (mask >> i) & 1]
        if sum(c) <= s:
            key = (-sum(c), -len(c), c)
            if best_key is None or key < best_key:
                best_key, best = key, c
    return best


assert answer_most_primes(15, 18) == [2, 3, 13]
assert answer_most_primes(7, 20) == [2, 3, 5, 7]
assert answer_most_primes(10, 16) == [3, 5, 7]
assert answer_most_primes(5, 1) == []
rng5 = random.Random(5)
for _ in range(150):
    n, s = rng5.randint(1, 35), rng5.randint(1, 150)
    assert answer_most_primes(n, s) == most_primes_bruteforce(n, s), (n, s)
print("Bài 5: PASS (kể cả 150 test ngẫu nhiên so với brute force)")

Bài 5: PASS (kể cả 150 test ngẫu nhiên so với brute force)


## Đáp án Bài 6 (mảng bất kỳ, có thể trùng)

Cách 3 không dùng tính chất "nguyên tố" ở đâu cả — chỉ cần các số **dương** và **đã sắp xếp tăng dần**. Với số trùng nhau (ví dụ hai số `5`), lập luận vẫn đúng: "chọn phần tử sớm nhất có thể" luôn cho giá trị nhỏ nhất ở vị trí đang xét.

In [40]:
def answer_general_subset(nums, s):
    a = sorted(nums)
    suf = build_suffix_table(a, s)          # dùng lại hàm của Cách 3
    T = 0
    for t in range(s, 0, -1):
        if suf[0][t] < INF:
            T = t
            break
    ans, rem = [], T
    for i in range(len(a)):
        if rem == 0:
            break
        if a[i] <= rem and suf[i + 1][rem - a[i]] + 1 == suf[i][rem]:
            ans.append(a[i])
            rem -= a[i]
    return ans


def general_bruteforce(nums, s):
    a = sorted(nums)
    m = len(a)
    best_key, best = None, []
    for mask in range(1, 1 << m):
        c = [a[i] for i in range(m) if (mask >> i) & 1]
        if sum(c) <= s:
            key = (-sum(c), len(c), c)
            if best_key is None or key < best_key:
                best_key, best = key, c
    return best


assert answer_general_subset([6, 4, 4, 1, 3], 9) == [3, 6]
assert answer_general_subset([5, 5, 2, 8], 10) == [2, 8]
assert answer_general_subset([2, 2, 3, 7], 7) == [7]
assert answer_general_subset([4, 4, 4], 3) == []
rng6 = random.Random(6)
for _ in range(300):
    nums = [rng6.randint(1, 15) for _ in range(rng6.randint(0, 9))]
    s = rng6.randint(1, 60)
    assert answer_general_subset(nums, s) == general_bruteforce(nums, s), (nums, s)
print("Bài 6: PASS (kể cả 300 test ngẫu nhiên so với brute force)")

Bài 6: PASS (kể cả 300 test ngẫu nhiên so với brute force)


# Tổng kết

Bài **Prime Subset Selection II** là bài tập tuyệt vời để học cách xử lý **nhiều tiêu chí ưu tiên** trên một bài knapsack:

1. **Tiêu chí 1 (tổng lớn nhất `≤ s`)** → subset sum: trạng thái là **tổng**, không phải tập. `2^m` tập con co lại thành `≤ s + 1` tổng.
2. **Tiêu chí 2 (ít phần tử nhất)** → DP "số lượng ít nhất" với `INF`.
3. **Tiêu chí 3 (nhỏ nhất theo từ điển)** → bảng **hậu tố** `suf[i][t]` + đi **từ trái sang phải**, chọn số nhỏ nhất mà vẫn giữ được tối ưu.

Hai cách tối ưu đều đạt yêu cầu: Cách 2 (lưu tuple, `O(m · s · K)`) ngắn gọn; Cách 3 (`O(m · s)`) là pattern tổng quát nên thuộc lòng.

## Template ngắn dễ nhớ

```python
primes = sieve(n)                                  # tăng dần
suf = [[INF] * (s + 1) for _ in range(m + 1)]
suf[m][0] = 0
for i in range(m - 1, -1, -1):                     # hậu tố ngắn -> dài
    for t in range(s + 1):
        suf[i][t] = min(suf[i + 1][t],
                        suf[i + 1][t - primes[i]] + 1 if t >= primes[i] else INF)

T = max((t for t in range(1, s + 1) if suf[0][t] < INF), default=0)   # T == 0 -> return []
rem = T
for i in range(m):                                 # trái -> phải
    if rem and primes[i] <= rem and suf[i + 1][rem - primes[i]] + 1 == suf[i][rem]:
        ans.append(primes[i]); rem -= primes[i]
```

Câu thần chú: **"Muốn chọn từ trái sang phải cho nhỏ nhất theo từ điển, hãy hỏi bảng hậu tố xem phần bên phải còn hoàn thành tối ưu được không."**

## Checklist trước khi submit

- [ ] Sàng bao gồm cả `n` (`range(..., n + 1)`) và **không** tính `1` là số nguyên tố.
- [ ] 0/1 knapsack: duyệt tổng **ngược** (bản 1 chiều) hoặc dùng bảng 2 chiều theo chỉ số (bản hậu tố).
- [ ] Thứ tự tiêu chí đúng: **tổng lớn nhất → ít phần tử nhất → từ điển**; khóa so sánh có `len` **trước** mảng.
- [ ] So sánh **list/tuple số**, không so chuỗi.
- [ ] Bảng hậu tố: `suf[m][0] = 0`, còn lại `INF`; tính `i` từ `m − 1` về `0`.
- [ ] Dựng đáp án **từ trái sang phải** với điều kiện `1 + suf[i+1][rem − p] == suf[i][rem]`.
- [ ] Tìm `T` trong `s..1` (không tính 0); không có → trả `[]`.
- [ ] Đã test: 3 ví dụ của đề, `n = 1`, `s = 1`, `n = 3, s = 4`, `n = 5, s = 6`, `n = 7, s = 8`, `n = s = 1000`.
- [ ] Đã so với brute force trên input nhỏ ngẫu nhiên.
- [ ] Đổi tên phương thức cho khớp khung code LeetCode thực tế.